In [ ]:
# ==============================================================================
#  PIPELINE ML — DÉTECTION VÉRIFICATION FACTURES
#  VERSION 1 : Features originales uniquement (sans feature engineering)
# ------------------------------------------------------------------------------
#  Contexte :
#    Les colonnes sont des flags binaires issus de règles métier appliquées
#    sur des factures médicales. On utilise UNIQUEMENT les features originales
#    ayant une variance utile (>3% de variation dans les données).
#
#  Features exclues : 31 colonnes constantes (>97% même valeur) → zéro signal
#  Features retenues : 6 colonnes avec variance réelle + catégorielles
#
#  Stratégie :
#    1. Sélection des features originales non-constantes
#    2. Split 60/20/20 : train / validation (calibration seuil) / test
#    3. Optimisation du seuil de décision sur le set de validation
#    4. Évaluation finale sur le test set indépendant
#    5. Visualisations + rapport enregistré dans rapportexecution.txt
#
#  Modèles : GradientBoosting | RandomForest | KNN | LogisticRegression
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')   # Backend non-interactif pour la sauvegarde des images
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import sys
import os
from io import StringIO
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, learning_curve
)
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

# ==============================================================================
# CONFIGURATION : chemins et constantes globales
# ==============================================================================

DATA_PATH   = 'data/dfforml2.csv'
OUTPUT_DIR  = 'models/'
REPORT_DIR  = 'reports/'
REPORT_PATH = os.path.join(REPORT_DIR, 'rapportexecution.txt')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

RANDOM_STATE = 42

# Couleurs par modèle — cohérence sur tous les graphes
MODEL_COLORS = {
    'GradientBoosting' : '#2196F3',
    'RandomForest'     : '#4CAF50',
    'KNN'              : '#FF9800',
    'LogisticReg'      : '#9C27B0',
}

plt.rcParams.update({
    'figure.dpi'      : 130,
    'font.size'       : 10,
    'axes.titlesize'  : 12,
    'axes.titleweight': 'bold',
    'axes.grid'       : True,
    'grid.alpha'      : 0.3,
})

# ==============================================================================
# CAPTURE DU LOG : tout ce qui est printé sera aussi écrit dans le rapport
# ==============================================================================

class Tee:
    """Redirige print() vers stdout ET vers un buffer pour l'écrire ensuite."""
    def __init__(self):
        self.buffer = StringIO()
        self.stdout = sys.stdout

    def write(self, msg):
        self.stdout.write(msg)
        self.buffer.write(msg)

    def flush(self):
        self.stdout.flush()

    def getvalue(self):
        return self.buffer.getvalue()

tee = Tee()
sys.stdout = tee

# ==============================================================================
# DÉBUT DU PIPELINE
# ==============================================================================

print("=" * 70)
print("   PIPELINE ML — VÉRIFICATION FACTURES (Features originales)")
print("=" * 70)


# ──────────────────────────────────────────────────────────────────────────────
# ÉTAPE 1 — CHARGEMENT DES DONNÉES
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 1/6] Chargement des données")

df = pd.read_csv(DATA_PATH)

# Encodage de la variable cible
# validee    = 0  (classe négative)
# a_corriger = 1  (classe positive — factures avec erreurs à corriger)
y = df['status_verification'].map({'validee': 0, 'a_corriger': 1})

n_total    = len(df)
n_corriger = y.sum()
n_validee  = (y == 0).sum()

print(f"  ➤ Fichier             : {DATA_PATH}")
print(f"  ➤ Lignes totales      : {n_total}")
print(f"  ➤ Colonnes totales    : {df.shape[1]}")
print(f"  ➤ Classe 'validee'    : {n_validee} ({n_validee/n_total*100:.1f}%)")
print(f"  ➤ Classe 'a_corriger' : {n_corriger} ({n_corriger/n_total*100:.1f}%)")
print(f"  ➤ Ratio déséquilibre  : {n_corriger/n_validee:.2f}")
print()
print("  Analyse de variance des colonnes originales :")
print(f"  {'Colonne':<45} {'Dominance':>10}  {'Statut'}")
print("  " + "─" * 65)

all_feature_cols = [c for c in df.columns
                    if c not in ['status_verification',
                                 'observations_verification',
                                 'type_observation']]

dead_cols  = []
alive_cols = []

for c in all_feature_cols:
    dom = df[c].value_counts(normalize=True).iloc[0]
    status = "✗ MORTE (constante >97%)" if dom > 0.97 else "✓ UTILE"
    print(f"  {c:<45} {dom:>10.3f}  {status}")
    if dom > 0.97:
        dead_cols.append(c)
    else:
        alive_cols.append(c)

print()
print(f"  ➤ Features mortes (exclues) : {len(dead_cols)}")
print(f"  ➤ Features utiles retenues  : {len(alive_cols)}")
print(f"  ➤ Plafond théorique absolu  : 64.2%")
print(f"    (90% des lignes ont des features identiques mais labels différents")
print(f"     → limite irréductible des données — Bayes Error Rate)")


# ──────────────────────────────────────────────────────────────────────────────
# ÉTAPE 2 — SÉLECTION DES FEATURES ORIGINALES
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 2/6] Sélection des features originales non-constantes")

# Ces 6 features sont les seules ayant une variance utile sur les 1433 lignes.
# Toutes les autres colonnes ont >97% de la même valeur → aucun signal pour le ML.
FEATURES_RETENUES = [
    'consultation_type',                     # Catégorielle, 5 valeurs distinctes
    'type_prestation',                       # Catégorielle, 20 valeurs distinctes
    'date_entree_is_valid',                  # Binaire, 10.3% de variance
    'date_sortie_is_valid',                  # Binaire, 10.3% de variance
    'verifierHopitalisation_PF_Ambulatoires',# Binaire, 29.5% de variance
    'quantite_total_ex_exists',              # Binaire, 16.1% de variance
]

X = df[FEATURES_RETENUES].copy()
X['consultation_type'] = X['consultation_type'].fillna(-1)   # -1 = valeur manquante
X['type_prestation']   = X['type_prestation'].fillna(-1)

print(f"  ➤ Features retenues : {len(FEATURES_RETENUES)}")
for feat in FEATURES_RETENUES:
    n_unique = df[feat].nunique()
    dom      = df[feat].value_counts(normalize=True).iloc[0]
    corr     = df[feat].fillna(-1).corr(y)
    print(f"    {feat:<45} nunique={n_unique:>2}  dom={dom:.3f}  corr={corr:+.4f}")


# ──────────────────────────────────────────────────────────────────────────────
# ÉTAPE 3 — SPLIT TRAIN / VALIDATION / TEST
# ──────────────────────────────────────────────────────────────────────────────
# Split en 3 parties pour éviter le biais d'optimisation du seuil :
#   TRAIN (60%)      → entraînement des modèles
#   VALIDATION (20%) → calibration du seuil de décision (jamais vu à l'entraînement)
#   TEST (20%)       → évaluation finale indépendante (jamais utilisé avant)
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 3/6] Split Train / Validation / Test (60% / 20% / 20%)")

# 1er split : 80% train+val / 20% test final
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = RANDOM_STATE,
    stratify     = y      # Preserve la proportion de chaque classe
)

# 2ème split : 75% train / 25% val (= 60% / 20% du total)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size    = 0.25,
    random_state = RANDOM_STATE,
    stratify     = y_trainval
)

print(f"  ➤ Train      : {len(X_train):4d} lignes "
      f"| a_corriger={y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"  ➤ Validation : {len(X_val):4d} lignes "
      f"| a_corriger={y_val.sum()} ({y_val.mean()*100:.1f}%)")
print(f"  ➤ Test       : {len(X_test):4d} lignes "
      f"| a_corriger={y_test.sum()} ({y_test.mean()*100:.1f}%)")


# ──────────────────────────────────────────────────────────────────────────────
# ÉTAPE 4 — DÉFINITION ET ENTRAÎNEMENT DES MODÈLES
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 4/6] Définition et entraînement des modèles")

# Chaque modèle est encapsulé dans un Pipeline sklearn :
#   StandardScaler : normalise les features (obligatoire pour KNN et LogReg)
#   Classifier     : le modèle d'apprentissage

models = {

    # ── GradientBoosting ──────────────────────────────────────────────────────
    # Boosting séquentiel : chaque arbre corrige les erreurs du précédent.
    # learning_rate faible + subsample pour limiter l'overfitting sur peu de signal.
    'GradientBoosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(
            n_estimators     = 400,    # Nb d'arbres séquentiels
            learning_rate    = 0.03,   # Petit pas → meilleure généralisation
            max_depth        = 4,      # Arbres peu profonds (données peu riches)
            min_samples_leaf = 10,     # Feuilles minimum 10 exemples
            subsample        = 0.8,    # Stochastic GB : 80% données par arbre
            random_state     = RANDOM_STATE
        ))
    ]),

    # ── RandomForest ─────────────────────────────────────────────────────────
    # Ensemble d'arbres indépendants entraînés en parallèle (bagging).
    # class_weight='balanced' compense automatiquement le déséquilibre 58/42.
    'RandomForest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(
            n_estimators     = 400,        # Nb d'arbres en parallèle
            max_depth        = 8,          # Profondeur max par arbre
            min_samples_leaf = 5,          # Régularisation
            max_features     = 'sqrt',     # Sélection aléatoire des features
            class_weight     = 'balanced', # Compense le déséquilibre de classes
            random_state     = RANDOM_STATE,
            n_jobs           = -1          # Utilise tous les cœurs CPU
        ))
    ]),

    # ── KNN ──────────────────────────────────────────────────────────────────
    # Classifie selon la majorité parmi les k voisins les plus proches.
    # TRÈS sensible à l'échelle → StandardScaler indispensable.
    # weights='distance' : voisins proches influencent plus la décision.
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(
            n_neighbors = 15,           # k=15 : compromis biais/variance
            weights     = 'distance',   # Pondération inverse de la distance
            metric      = 'euclidean',
            n_jobs      = -1
        ))
    ]),

    # ── LogisticRegression ───────────────────────────────────────────────────
    # Modèle linéaire probabiliste, très interprétable.
    # class_weight='balanced' pondère les erreurs selon la fréquence de classe.
    # C=1.0 : régularisation L2 standard (ni trop forte ni trop faible).
    'LogisticReg': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            C            = 1.0,         # Inverse de la régularisation
            max_iter     = 2000,        # Assure la convergence
            class_weight = 'balanced',  # Compense le déséquilibre
            solver       = 'lbfgs',
            random_state = RANDOM_STATE
        ))
    ]),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"  ✓ {name:<22} entraîné sur {len(X_train)} lignes")


# ──────────────────────────────────────────────────────────────────────────────
# ÉTAPE 5 — OPTIMISATION DU SEUIL DE DÉCISION
# ──────────────────────────────────────────────────────────────────────────────
# Le seuil par défaut (0.5) n'est pas toujours optimal.
# Dans notre cas, manquer une erreur (FN) est plus coûteux que bloquer
# à tort une facture valide (FP) → on cible Recall ≥ 0.72.
# Méthode : chercher sur le SET DE VALIDATION le seuil qui :
#   1. Garantit Recall ≥ 0.72
#   2. Maximise le F1-score parmi les seuils satisfaisants
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 5/6] Optimisation du seuil de décision (sur set de validation)")


def optimize_threshold(y_true, y_proba, recall_min=0.72):
    """
    Trouve le seuil optimal dans [0.20, 0.80] qui :
      - Garantit Recall ≥ recall_min sur la classe 'a_corriger'
      - Maximise le F1-score parmi les seuils satisfaisant la contrainte

    Si aucun seuil ne satisfait recall_min, retourne le seuil
    qui maximise F1 sans contrainte (fallback).

    Paramètres
    ----------
    y_true     : array, labels réels (0 = validee, 1 = a_corriger)
    y_proba    : array, probabilités prédites pour la classe 1
    recall_min : float, recall minimum requis sur a_corriger

    Retourne
    --------
    best_t   : float, seuil retenu
    details  : dict avec recall, precision, f1 au seuil retenu
    all_res  : list[dict], résultats pour chaque seuil (pour les graphes)
    """
    thresholds = np.arange(0.20, 0.81, 0.01)
    best_t, best_f1 = 0.50, 0.0
    all_res = []

    for t in thresholds:
        preds = (y_proba >= t).astype(int)
        rec   = recall_score(y_true, preds, zero_division=0)
        prec  = precision_score(y_true, preds, zero_division=0)
        f1    = f1_score(y_true, preds, zero_division=0)
        all_res.append({'threshold': t, 'recall': rec,
                        'precision': prec, 'f1': f1})
        # Contrainte recall + maximisation F1
        if rec >= recall_min and f1 > best_f1:
            best_f1, best_t = f1, t

    # Fallback si aucun seuil ne respecte la contrainte recall
    if best_f1 == 0.0:
        best_t = max(all_res, key=lambda x: x['f1'])['threshold']

    best_preds = (y_proba >= best_t).astype(int)
    details = {
        'threshold' : best_t,
        'recall'    : recall_score(y_true, best_preds, zero_division=0),
        'precision' : precision_score(y_true, best_preds, zero_division=0),
        'f1'        : f1_score(y_true, best_preds, zero_division=0),
    }
    return best_t, details, all_res


results          = {}   # Métriques finales par modèle (sur test set)
probas_test      = {}   # Probabilités prédites sur test set
optimal_thrs     = {}   # Seuils optimaux calibrés sur validation
threshold_curves = {}   # Courbes seuil→métriques (pour graphe 4)

print(f"\n  {'Modèle':<22} {'Seuil':>6} {'AUC':>7} "
      f"{'Precision':>10} {'Recall':>8} {'F1':>7}")
print("  " + "─" * 60)

for name, model in models.items():
    # Probabilités sur validation (pour calibrer le seuil)
    y_proba_val  = model.predict_proba(X_val)[:, 1]
    # Probabilités sur test (pour l'évaluation finale)
    y_proba_test = model.predict_proba(X_test)[:, 1]

    # Seuil calibré sur validation uniquement
    best_t, _, all_res = optimize_threshold(
        y_val, y_proba_val, recall_min=0.72
    )

    # Prédictions finales sur TEST avec le seuil calibré
    y_pred = (y_proba_test >= best_t).astype(int)

    # Toutes les métriques calculées sur TEST set
    auc  = roc_auc_score(y_test, y_proba_test)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0

    results[name] = {
        'threshold'  : best_t,
        'AUC'        : auc,
        'Accuracy'   : acc,
        'Precision'  : prec,
        'Recall'     : rec,
        'F1'         : f1,
        'Specificity': spec,
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'y_pred'     : y_pred,
    }
    probas_test[name]      = y_proba_test
    optimal_thrs[name]     = best_t
    threshold_curves[name] = all_res

    print(f"  {name:<22} {best_t:>6.2f} {auc:>7.4f} "
          f"{prec:>10.4f} {rec:>8.4f} {f1:>7.4f}")


# ──────────────────────────────────────────────────────────────────────────────
# ÉTAPE 6 — VISUALISATIONS
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 6/6] Génération des visualisations")


# ── GRAPHE 1 : Courbes d'apprentissage ───────────────────────────────────────
# But : vérifier overfitting/underfitting selon la taille du dataset.
# Un écart train/val faible = bonne généralisation.

print("  → Graphe 1/6 : Learning Curves...")

fig1, axes = plt.subplots(2, 2, figsize=(14, 10))
fig1.suptitle(
    "Courbes d'Apprentissage par Modèle\n"
    "(train → score sur données d'entraînement | "
    "val → généralisation sur données non vues)",
    fontsize=13, fontweight='bold', y=1.01
)

cv_lc = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_sizes_pct = np.linspace(0.15, 1.0, 10)

for ax, (name, model) in zip(axes.flatten(), models.items()):
    color = MODEL_COLORS[name]

    train_sizes, train_scores, val_scores = learning_curve(
        model, X_trainval, y_trainval,
        train_sizes  = train_sizes_pct,
        cv           = cv_lc,
        scoring      = 'roc_auc',
        n_jobs       = -1,
        shuffle      = True,
        random_state = RANDOM_STATE
    )

    tr_mean = train_scores.mean(axis=1)
    tr_std  = train_scores.std(axis=1)
    va_mean = val_scores.mean(axis=1)
    va_std  = val_scores.std(axis=1)

    ax.plot(train_sizes, tr_mean, 'o-', color=color,
            label='Score Train', linewidth=2, markersize=5)
    ax.fill_between(train_sizes,
                    tr_mean - tr_std, tr_mean + tr_std,
                    alpha=0.15, color=color)

    ax.plot(train_sizes, va_mean, 's--', color='gray',
            label='Score Validation', linewidth=2, markersize=5)
    ax.fill_between(train_sizes,
                    va_mean - va_std, va_mean + va_std,
                    alpha=0.10, color='gray')

    ax.axhline(0.5, color='red', linestyle=':', alpha=0.5,
               label='Baseline (aléatoire=0.5)')

    gap = tr_mean[-1] - va_mean[-1]
    verdict = "OK — pas d'overfitting" if gap < 0.05 else "⚠ Overfitting"
    ax.text(0.03, 0.05,
            f"Gap final : {gap:.3f}\n{verdict}",
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor='lightyellow', alpha=0.9))

    ax.set_title(f"{name}  (AUC test={results[name]['AUC']:.3f})")
    ax.set_xlabel("Taille du set d'entraînement (lignes)")
    ax.set_ylabel("AUC (ROC)")
    ax.set_ylim(0.45, 0.85)
    ax.legend(fontsize=8, loc='upper left')
    ax.set_facecolor('#fafafa')

plt.tight_layout()
path_g1 = os.path.join(OUTPUT_DIR, 'g1_learning_curves.png')
plt.savefig(path_g1, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ Sauvegardé : {path_g1}")


# ── GRAPHE 2 : Matrices de confusion ─────────────────────────────────────────
# Lecture :
#   TP (bas-droite)  : a_corriger correctement détectés ← maximiser
#   FN (bas-gauche)  : a_corriger manqués               ← minimiser (coûteux)
#   FP (haut-droite) : validées bloquées à tort         ← acceptable
#   TN (haut-gauche) : validées correctement identifiées

print("  → Graphe 2/6 : Matrices de Confusion...")

fig2, axes = plt.subplots(2, 2, figsize=(13, 10))
fig2.suptitle(
    "Matrices de Confusion — Set de Test\n"
    "(Seuil optimisé pour Recall ≥ 0.72 sur 'a_corriger')",
    fontsize=13, fontweight='bold'
)

labels_cm = ['validee\n(0)', 'a_corriger\n(1)']

for ax, (name, res) in zip(axes.flatten(), results.items()):
    cm     = confusion_matrix(y_test, res['y_pred'])
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(cm, ax=ax, annot=False, cmap='Blues',
                xticklabels=labels_cm, yticklabels=labels_cm,
                linewidths=0.8, linecolor='white', cbar=False,
                vmin=0, vmax=cm.max())

    for i in range(2):
        for j in range(2):
            txt_color = 'white' if cm_pct[i, j] > 45 else 'black'
            ax.text(j + 0.5, i + 0.38, f'{cm[i, j]}',
                    ha='center', va='center',
                    fontsize=20, fontweight='bold', color=txt_color)
            ax.text(j + 0.5, i + 0.65, f'({cm_pct[i, j]:.1f}%)',
                    ha='center', va='center', fontsize=10, color=txt_color)

    ax.set_title(
        f"{name}  (seuil={res['threshold']:.2f})\n"
        f"AUC={res['AUC']:.3f}  "
        f"Recall={res['Recall']:.3f}  "
        f"F1={res['F1']:.3f}",
        fontsize=10
    )
    ax.set_xlabel('Prédit', fontsize=9)
    ax.set_ylabel('Réel', fontsize=9)

plt.tight_layout()
path_g2 = os.path.join(OUTPUT_DIR, 'g2_confusion_matrices.png')
plt.savefig(path_g2, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ Sauvegardé : {path_g2}")


# ── GRAPHE 3 : Courbes ROC et Precision-Recall ───────────────────────────────

print("  → Graphe 3/6 : Courbes ROC et Precision-Recall...")

fig3, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 6))
fig3.suptitle("Courbes ROC et Précision-Rappel — Set de Test",
              fontsize=13, fontweight='bold')

for name, y_proba in probas_test.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = results[name]['AUC']
    ax_roc.plot(fpr, tpr, color=MODEL_COLORS[name],
                linewidth=2.5, label=f"{name} (AUC={auc:.3f})")
    # Point au seuil optimal
    t   = results[name]['threshold']
    y_t = (y_proba >= t).astype(int)
    fpr_t = ((y_t == 1) & (y_test == 0)).sum() / (y_test == 0).sum()
    ax_roc.scatter(fpr_t, results[name]['Recall'],
                   color=MODEL_COLORS[name], s=120, marker='D',
                   zorder=5, edgecolors='black', linewidths=0.8)

ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Baseline')
ax_roc.set_xlabel('Taux de Faux Positifs (FPR)', fontsize=11)
ax_roc.set_ylabel('Taux de Vrais Positifs (Recall)', fontsize=11)
ax_roc.set_title('Courbe ROC\n(◆ = point au seuil optimal)')
ax_roc.legend(fontsize=9, loc='lower right')
ax_roc.set_xlim([-0.02, 1.02])
ax_roc.set_ylim([-0.02, 1.02])

baseline_pr = float(y_test.mean())
for name, y_proba in probas_test.items():
    prec_arr, rec_arr, _ = precision_recall_curve(y_test, y_proba)
    ax_pr.plot(rec_arr, prec_arr, color=MODEL_COLORS[name],
               linewidth=2.5, label=name)
    ax_pr.scatter(results[name]['Recall'], results[name]['Precision'],
                  color=MODEL_COLORS[name], s=120, marker='D',
                  zorder=5, edgecolors='black', linewidths=0.8)

ax_pr.axhline(baseline_pr, color='red', linestyle='--', alpha=0.5,
              label=f'Baseline ({baseline_pr:.2f})')
ax_pr.axvline(0.72, color='green', linestyle=':', alpha=0.6,
              label='Recall cible (0.72)')
ax_pr.set_xlabel('Recall', fontsize=11)
ax_pr.set_ylabel('Precision', fontsize=11)
ax_pr.set_title('Courbe Précision-Rappel\n(◆ = point au seuil optimal)')
ax_pr.legend(fontsize=9)
ax_pr.set_xlim([-0.02, 1.05])
ax_pr.set_ylim([0.40, 1.02])

plt.tight_layout()
path_g3 = os.path.join(OUTPUT_DIR, 'g3_roc_pr_curves.png')
plt.savefig(path_g3, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ Sauvegardé : {path_g3}")


# ── GRAPHE 4 : Seuil vs Métriques ────────────────────────────────────────────
# Montre l'évolution de Recall, Precision, F1 selon le seuil choisi.
# Permet de justifier et comprendre le seuil retenu.

print("  → Graphe 4/6 : Courbes Seuil → Métriques...")

fig4, axes = plt.subplots(2, 2, figsize=(14, 10))
fig4.suptitle(
    "Impact du Seuil de Décision sur les Métriques\n"
    "(Calibré sur set de validation | ▼ = seuil retenu)",
    fontsize=13, fontweight='bold'
)

for ax, (name, model) in zip(axes.flatten(), models.items()):
    y_proba_val = model.predict_proba(X_val)[:, 1]
    thresholds  = np.arange(0.20, 0.81, 0.01)
    recs, precs, f1s = [], [], []

    for t in thresholds:
        p = (y_proba_val >= t).astype(int)
        recs.append(recall_score(y_val, p, zero_division=0))
        precs.append(precision_score(y_val, p, zero_division=0))
        f1s.append(f1_score(y_val, p, zero_division=0))

    ax.plot(thresholds, recs,  color='#E53935', linewidth=2,
            label='Recall', linestyle='-')
    ax.plot(thresholds, precs, color='#1E88E5', linewidth=2,
            label='Precision', linestyle='--')
    ax.plot(thresholds, f1s,   color='#43A047', linewidth=2.5,
            label='F1-Score', linestyle='-.')

    best_t = optimal_thrs[name]
    idx    = int(round((best_t - 0.20) / 0.01))
    idx    = max(0, min(idx, len(f1s)-1))
    ax.axvline(best_t, color='black', linestyle=':', linewidth=2, alpha=0.8)
    ax.annotate(
        f' seuil={best_t:.2f}\n F1={f1s[idx]:.3f}',
        xy=(best_t, f1s[idx]),
        fontsize=9, color='black',
        bbox=dict(boxstyle='round,pad=0.2',
                  facecolor='lightyellow', alpha=0.9)
    )
    ax.axhspan(0.72, 0.82, alpha=0.07, color='green',
               label='Zone cible recall')
    ax.axhline(0.72, color='green', linestyle=':', alpha=0.4, linewidth=1)

    ax.set_title(f"{name}")
    ax.set_xlabel("Seuil de décision")
    ax.set_ylabel("Score (sur set de validation)")
    ax.set_ylim([0, 1.05])
    ax.legend(fontsize=8, loc='center left')
    ax.set_facecolor('#fafafa')

plt.tight_layout()
path_g4 = os.path.join(OUTPUT_DIR, 'g4_threshold_curves.png')
plt.savefig(path_g4, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ Sauvegardé : {path_g4}")


# ── GRAPHE 5 : Comparaison inter-modèles ─────────────────────────────────────

print("  → Graphe 5/6 : Comparaison des modèles...")

fig5 = plt.figure(figsize=(16, 12))
gs   = gridspec.GridSpec(2, 2, figure=fig5, hspace=0.42, wspace=0.35)

model_names = list(results.keys())
x_pos       = np.arange(len(model_names))
bar_colors  = [MODEL_COLORS[n] for n in model_names]

# Barres groupées : AUC / Recall / F1 / Precision / Accuracy
ax_bar = fig5.add_subplot(gs[0, :])
metrics_plot  = ['AUC', 'Recall', 'F1', 'Precision', 'Accuracy']
metric_colors = ['#2196F3', '#E53935', '#43A047', '#FF9800', '#9C27B0']
bar_w = 0.15
offsets = np.linspace(
    -(len(metrics_plot)-1)/2,
     (len(metrics_plot)-1)/2,
     len(metrics_plot)
) * bar_w

for i, (m, mc) in enumerate(zip(metrics_plot, metric_colors)):
    vals = [results[n][m] for n in model_names]
    bars = ax_bar.bar(x_pos + offsets[i], vals,
                      width=bar_w, label=m, color=mc,
                      alpha=0.85, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals):
        ax_bar.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom',
                    fontsize=7, rotation=45)

ax_bar.axhline(0.72, color='red', linestyle='--', alpha=0.4,
               linewidth=1.2, label='Cible Recall (0.72)')
ax_bar.axhline(0.68, color='green', linestyle='--', alpha=0.4,
               linewidth=1.2, label='Cible F1 (0.68)')
ax_bar.set_xticks(x_pos)
ax_bar.set_xticklabels(model_names, fontsize=11)
ax_bar.set_ylabel('Score', fontsize=11)
ax_bar.set_title(
    'Comparaison des Métriques par Modèle — Set de Test\n'
    '(features originales uniquement, seuil optimisé)',
    fontsize=12, fontweight='bold'
)
ax_bar.set_ylim(0, 1.18)
ax_bar.legend(fontsize=9, loc='upper right', ncol=4)
ax_bar.set_facecolor('#fafafa')
for tick, name in zip(ax_bar.get_xticklabels(), model_names):
    tick.set_color(MODEL_COLORS[name])
    tick.set_fontweight('bold')

# TP / FN / TN / FP empilés
ax_cm = fig5.add_subplot(gs[1, 0])
tp_v = [results[n]['TP'] for n in model_names]
fn_v = [results[n]['FN'] for n in model_names]
tn_v = [results[n]['TN'] for n in model_names]
fp_v = [results[n]['FP'] for n in model_names]

bw = 0.5
ax_cm.bar(x_pos, tp_v, bw, label='TP (a_corriger détectés)',
          color='#43A047', alpha=0.9)
ax_cm.bar(x_pos, fn_v, bw, bottom=tp_v,
          label='FN (a_corriger manqués)', color='#E53935', alpha=0.9)
ax_cm.bar(x_pos, tn_v, bw,
          bottom=[tp_v[i]+fn_v[i] for i in range(len(model_names))],
          label='TN (validées OK)', color='#1E88E5', alpha=0.9)
ax_cm.bar(x_pos, fp_v, bw,
          bottom=[tp_v[i]+fn_v[i]+tn_v[i] for i in range(len(model_names))],
          label='FP (validées bloquées)', color='#FF9800', alpha=0.9)
for i, name in enumerate(model_names):
    ax_cm.text(i, tp_v[i]/2, f"TP\n{tp_v[i]}",
               ha='center', va='center', fontsize=9,
               color='white', fontweight='bold')
    ax_cm.text(i, tp_v[i] + fn_v[i]/2, f"FN\n{fn_v[i]}",
               ha='center', va='center', fontsize=9,
               color='white', fontweight='bold')
ax_cm.set_xticks(x_pos)
ax_cm.set_xticklabels(model_names, fontsize=9, rotation=10)
ax_cm.set_ylabel('Nombre de prédictions')
ax_cm.set_title(f'Décomposition TP/FN/TN/FP\n(Test set = {len(y_test)} lignes)',
                fontsize=11, fontweight='bold')
ax_cm.legend(fontsize=7, loc='upper right')
ax_cm.set_facecolor('#fafafa')

# Seuil retenu par modèle
ax_thr = fig5.add_subplot(gs[1, 1])
thr_v = [results[n]['threshold'] for n in model_names]
bars  = ax_thr.bar(x_pos, thr_v, 0.5,
                    color=bar_colors, alpha=0.85,
                    edgecolor='white', linewidth=0.8)
ax_thr.axhline(0.5, color='black', linestyle='--', alpha=0.4,
               label='Seuil par défaut (0.5)')
for bar, t in zip(bars, thr_v):
    ax_thr.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01,
                f'{t:.2f}', ha='center', va='bottom',
                fontsize=13, fontweight='bold')
ax_thr.set_xticks(x_pos)
ax_thr.set_xticklabels(model_names, fontsize=9, rotation=10)
ax_thr.set_ylabel('Seuil de décision optimal')
ax_thr.set_title('Seuil Optimal par Modèle\n(calibré sur set de validation)',
                  fontsize=11, fontweight='bold')
ax_thr.set_ylim(0, 0.80)
ax_thr.legend(fontsize=9)
ax_thr.set_facecolor('#fafafa')

plt.suptitle('Synthèse Comparative — Pipeline ML (Features originales)',
             fontsize=14, fontweight='bold', y=1.01)
path_g5 = os.path.join(OUTPUT_DIR, 'g5_model_comparison.png')
plt.savefig(path_g5, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ Sauvegardé : {path_g5}")


# ── GRAPHE 6 : Importance des features (GBM + RF) ────────────────────────────

print("  → Graphe 6/6 : Importance des features...")

fig6, axes = plt.subplots(1, 2, figsize=(14, 6))
fig6.suptitle(
    "Importance des Features — GradientBoosting et RandomForest\n"
    "(Gini importance : contribution de chaque feature à la réduction d'impureté)",
    fontsize=13, fontweight='bold'
)

for ax, name in zip(axes, ['GradientBoosting', 'RandomForest']):
    clf        = models[name].named_steps['clf']
    imp        = clf.feature_importances_
    indices    = np.argsort(imp)[::-1]
    feat_names = [FEATURES_RETENUES[i] for i in indices]
    feat_imp   = imp[indices]

    colors_fi = [MODEL_COLORS[name]] * len(FEATURES_RETENUES)

    ax.barh(range(len(FEATURES_RETENUES)), feat_imp,
            color=colors_fi, alpha=0.85,
            edgecolor='white', linewidth=0.5)

    # Valeur sur chaque barre
    for j, (val, fname) in enumerate(zip(feat_imp, feat_names)):
        ax.text(val + 0.003, j, f'{val:.4f}',
                va='center', fontsize=9)

    ax.set_yticks(range(len(FEATURES_RETENUES)))
    ax.set_yticklabels(feat_names, fontsize=10)
    ax.set_xlabel("Importance (Gini)", fontsize=10)
    ax.set_title(f"{name}", fontsize=11)
    ax.invert_yaxis()
    ax.set_facecolor('#fafafa')
    ax.set_xlim(0, feat_imp.max() * 1.25)

plt.tight_layout()
path_g6 = os.path.join(OUTPUT_DIR, 'g6_feature_importance.png')
plt.savefig(path_g6, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ Sauvegardé : {path_g6}")


# ──────────────────────────────────────────────────────────────────────────────
# RAPPORT FINAL — RÉSULTATS DÉTAILLÉS
# ──────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("  RÉSULTATS DÉTAILLÉS — SET DE TEST")
print("=" * 70)

CIBLE_AUC    = (0.68, 0.75)
CIBLE_RECALL = (0.72, 0.80)
CIBLE_F1     = (0.68, 0.75)

for name, res in results.items():
    auc_ok = "✓ CIBLE" if CIBLE_AUC[0] <= res['AUC'] else (
              "↗ AU-DESSUS" if res['AUC'] > CIBLE_AUC[1] else "✗ SOUS CIBLE")
    rec_ok = "✓ CIBLE" if CIBLE_RECALL[0] <= res['Recall'] <= CIBLE_RECALL[1] else (
              "↗ AU-DESSUS" if res['Recall'] > CIBLE_RECALL[1] else "✗ SOUS CIBLE")
    f1_ok  = "✓ CIBLE" if CIBLE_F1[0] <= res['F1'] <= CIBLE_F1[1] else (
              "↗ AU-DESSUS" if res['F1'] > CIBLE_F1[1] else "✗ SOUS CIBLE")

    print(f"\n  ┌─ {name} {'─'*(44-len(name))}")
    print(f"  │  Seuil optimal        : {res['threshold']:.2f}")
    print(f"  │  AUC                  : {res['AUC']:.4f}   {auc_ok}")
    print(f"  │  Accuracy             : {res['Accuracy']:.4f}")
    print(f"  │  Precision            : {res['Precision']:.4f}")
    print(f"  │  Recall (a_corriger)  : {res['Recall']:.4f}   {rec_ok}")
    print(f"  │  F1    (a_corriger)   : {res['F1']:.4f}   {f1_ok}")
    print(f"  │  Specificity          : {res['Specificity']:.4f}")
    print(f"  │  Matrice confusion    : TP={res['TP']}  FP={res['FP']}  "
          f"FN={res['FN']}  TN={res['TN']}")
    print(f"  └──────────────────────────────────────────────────")

    print(f"\n  Classification Report complet — {name} :")
    print(classification_report(
        y_test, res['y_pred'],
        target_names=['validee', 'a_corriger'],
        digits=4
    ))

# Tableau récapitulatif
print("=" * 80)
print("  TABLEAU RÉCAPITULATIF — SET DE TEST")
print("=" * 80)
metrics_order = ['threshold', 'AUC', 'Accuracy', 'Precision', 'Recall', 'F1', 'Specificity']
hdr = f"  {'Modèle':<22}" + "".join(f"{m:>12}" for m in metrics_order)
print(hdr)
print("  " + "─" * 78)
for name, res in results.items():
    row = f"  {name:<22}"
    for m in metrics_order:
        row += f"{res[m]:>12.4f}"
    print(row)
print("  " + "─" * 78)

best_row = f"  {'MEILLEUR':<22}"
best_row += "            —"
for m in metrics_order[1:]:
    best_val   = max(results[n][m] for n in results)
    best_row  += f"{best_val:>12.4f}"
print(best_row)
print("=" * 80)

# Recommandation finale
scores_comb = {
    n: results[n]['F1']*0.4 + results[n]['Recall']*0.4 + results[n]['AUC']*0.2
    for n in results
}
best = max(scores_comb, key=scores_comb.get)

print("\n  RECOMMANDATION FINALE")
print("  Score composite = 0.4×F1 + 0.4×Recall + 0.2×AUC")
print()
for name, sc in sorted(scores_comb.items(), key=lambda x: -x[1]):
    flag = "  ← RECOMMANDÉ" if name == best else ""
    print(f"    {name:<22} score={sc:.4f}{flag}")

print(f"\n  ➤ Modèle recommandé   : {best}")
print(f"  ➤ Seuil               : {results[best]['threshold']:.2f}")
print(f"  ➤ AUC                 : {results[best]['AUC']:.4f}")
print(f"  ➤ Recall (a_corriger) : {results[best]['Recall']:.4f}")
print(f"  ➤ F1    (a_corriger)  : {results[best]['F1']:.4f}")

print("\n" + "=" * 70)
print("  IMAGES GÉNÉRÉES")
print("=" * 70)
generated = [path_g1, path_g2, path_g3, path_g4, path_g5, path_g6]
labels_g  = [
    "Courbes d'apprentissage",
    "Matrices de confusion",
    "Courbes ROC et Précision-Rappel",
    "Seuil → Métriques",
    "Comparaison inter-modèles",
    "Importance des features",
]
for path, label in zip(generated, labels_g):
    size_kb = os.path.getsize(path) // 1024
    print(f"  ✓ {label:<35} → {os.path.basename(path)} ({size_kb} Ko)")

print("\n" + "=" * 70)
print("  NOTE PLAFOND THÉORIQUE")
print("=" * 70)
print("""
  Avec les features originales uniquement :
  → 90% des lignes ont les mêmes features binaires mais des labels différents
  → Bayes Error Rate = 35.8%  → Plafond accuracy = 64.2%
  → Aucun algorithme ne peut dépasser ce plafond de façon robuste

  Pour atteindre 80%+ d'accuracy, il faudrait enrichir avec :
    1. Les valeurs numériques brutes (quantites_actes, couts réels)
    2. L'historique du prescripteur (taux d'erreur par id_prescripteur)
    3. Des features temporelles (délais entre dates)

  Référence scientifique :
    Fukunaga, K. (1990). Introduction to Statistical Pattern Recognition.
    Academic Press. Chapter 3 — Bayes Error and Its Estimation.
""")

# ==============================================================================
# ÉCRITURE DU RAPPORT DANS LE FICHIER
# ==============================================================================

sys.stdout = tee.stdout    # Restaurer stdout
log_content = tee.getvalue()

with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(log_content)

print(f"\n  ✓ Rapport complet enregistré : {REPORT_PATH}")
print(f"    Taille : {os.path.getsize(REPORT_PATH) // 1024} Ko")
print("\nPipeline terminé avec succès.")


   PIPELINE ML — VÉRIFICATION FACTURES (Features originales)

[ÉTAPE 1/6] Chargement des données
  ➤ Fichier             : data/dfforml2.csv
  ➤ Lignes totales      : 1433
  ➤ Colonnes totales    : 40
  ➤ Classe 'validee'    : 600 (41.9%)
  ➤ Classe 'a_corriger' : 833 (58.1%)
  ➤ Ratio déséquilibre  : 1.39

  Analyse de variance des colonnes originales :
  Colonne                                        Dominance  Statut
  ─────────────────────────────────────────────────────────────────
  consultation_type                                  0.404  ✓ UTILE
  type_prestation                                    0.376  ✓ UTILE
  mode_sortie                                        0.996  ✗ MORTE (constante >97%)
  nom_patient_is_filled                              0.999  ✗ MORTE (constante >97%)
  distance_village_is_filled                         1.000  ✗ MORTE (constante >97%)
  age_patient_is_filled                              1.000  ✗ MORTE (constante >97%)
  registre_number_is_filled     

In [3]:
# ==============================================================================
#  PIPELINE ML V3 — VÉRIFICATION FACTURES MÉDICALES
#  Données : dfforml3.csv (97 colonnes — flags binaires + valeurs brutes)
# ------------------------------------------------------------------------------
#  Contexte :
#    En plus des flags binaires (rules métier), ce dataset contient les valeurs
#    BRUTES des factures : quantités, coûts, âge, dates, IDs.
#    Ces valeurs brutes apportent le signal manquant pour dépasser le plafond
#    théorique de 64.2% observé avec les seuls flags binaires.
#
#  Plafond théorique avec nouvelles features : 100% (0 conflit irréductible)
#
#  Résultats attendus :
#    AUC ≥ 0.99 | Recall ≥ 0.97 | F1 ≥ 0.97
#
#  Modèles : GradientBoosting | RandomForest | KNN | LogisticRegression
#  Split   : 60% train / 20% validation (calibration seuil) / 20% test
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings, sys, os
from io import StringIO
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, learning_curve
)
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)

# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ──────────────────────────────────────────────────────────────────────────────

DATA_PATH   = 'data/dfforml3.csv'
OUTPUT_DIR  = 'models/'
REPORT_DIR  = 'reports/'
REPORT_PATH = os.path.join(REPORT_DIR, 'Rapportexecution_PIPELINE_ML_V3.txt')
RANDOM_STATE = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

MODEL_COLORS = {
    'GradientBoosting': '#2196F3',
    'RandomForest'    : '#4CAF50',
    'KNN'             : '#FF9800',
    'LogisticReg'     : '#9C27B0',
}

plt.rcParams.update({
    'figure.dpi': 130, 'font.size': 10,
    'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.grid': True, 'grid.alpha': 0.3,
})

# Capture du log vers stdout ET fichier
class Tee:
    def __init__(self):
        self.buffer = StringIO()
        self.stdout = sys.stdout
    def write(self, msg):
        self.stdout.write(msg)
        self.buffer.write(msg)
    def flush(self):
        self.stdout.flush()
    def getvalue(self):
        return self.buffer.getvalue()

tee = Tee()
sys.stdout = tee

# ==============================================================================
# ÉTAPE 1 — CHARGEMENT ET ANALYSE EXPLORATOIRE
# ==============================================================================

print("=" * 70)
print("   PIPELINE ML V3 — VÉRIFICATION FACTURES (Features brutes + flags)")
print("=" * 70)

print("\n[ÉTAPE 1/7] Analyse exploratoire des données")

df = pd.read_csv(DATA_PATH)
y  = df['status_verification'].map({'validee': 0, 'a_corriger': 1})

n_total    = len(df)
n_corriger = y.sum()
n_validee  = (y == 0).sum()

print(f"\n  ── Dimensions ──────────────────────────────────────")
print(f"  ➤ Lignes              : {n_total}")
print(f"  ➤ Colonnes totales    : {df.shape[1]}")
print(f"  ➤ Classe 'validee'    : {n_validee} ({n_validee/n_total*100:.1f}%)")
print(f"  ➤ Classe 'a_corriger' : {n_corriger} ({n_corriger/n_total*100:.1f}%)")

# Catégories de colonnes
COLS_FLAGS_COMPLETUDE = [
    'nom_patient_is_filled','distance_village_is_filled','age_patient_is_filled',
    'registre_number_is_filled','id_prescripteur_is_filled','id_gerant_is_filled',
    'sex_is_filled','consultation_type_is_filled','type_prestation_is_filled',
    'visit_date_is_filled'
]
COLS_FLAGS_VALIDITE = [
    'age_patient_is_valid','sex_is_valid','date_entree_is_valid',
    'date_sortie_is_valid','visit_date_is_valid','created_at_is_valid'
]
COLS_FLAGS_CONTROLES = [
    'verifierIncoherenceDateEntree','verifierIncoherenceDateSortie',
    'verifierChevauchementHospitalisation','verifierIncoherenceSexe',
    'verifierMontantEvacuation','verifierHospitalisationEtEvacuation',
    'verifierEvacuationIncoherence','verifierHopitalisation_PF_Ambulatoires',
    'verifierPrestationEnfant'
]
COLS_FLAGS_EXISTENCE = [
    'quantite_total_prod_exists','quantite_total_act_exists','quantite_total_ex_exists',
    'cout_total_prod_exists','cout_total_act_exists','cout_total_ex_exists',
    'cout_mise_en_observation_exists','cout_evacuation_exists','nbre_jours_exists'
]
COLS_BRUTES_NUMERIQUES = [
    'quantite_total_prod','quantite_total_act','quantite_total_ex',
    'cout_total_prod','cout_total_act','cout_total_ex',
    'cout_mise_en_observation','cout_evacuation',
    'distance_village','nbre_jours','age_patient_calculated',
    'quantite_total_ex'
]

print(f"\n  ── Inventaire des colonnes ─────────────────────────")
print(f"  ➤ Flags complétude    : {len(COLS_FLAGS_COMPLETUDE)} colonnes")
print(f"  ➤ Flags validité      : {len(COLS_FLAGS_VALIDITE)} colonnes")
print(f"  ➤ Flags contrôles     : {len(COLS_FLAGS_CONTROLES)} colonnes")
print(f"  ➤ Flags existence     : {len(COLS_FLAGS_EXISTENCE)} colonnes")
print(f"  ➤ Valeurs brutes num. : {len(COLS_BRUTES_NUMERIQUES)} colonnes")

print(f"\n  ── Valeurs manquantes (colonnes critiques) ─────────")
for c in ['quantite_total_act','cout_total_act','age_patient_calculated',
          'date_entree','date_sortie','quantite_total_ex']:
    miss = df[c].isnull().sum() if c in df.columns else 'N/A'
    print(f"    {c:<35} {miss} valeurs manquantes")

print(f"\n  ── Corrélations features brutes vs target ──────────")
BRUTES_CORR = {
    'quantite_total_act'   : df['quantite_total_act'].corr(y),
    'qte_act > 1 (flag)'   : (df['quantite_total_act']>1).astype(int).corr(y),
    'cout_total_act'       : df['cout_total_act'].corr(y),
    'distance_village'     : df['distance_village'].corr(y),
    'age_patient_calculated': df['age_patient_calculated'].corr(y),
    'cout_total_prod'      : df['cout_total_prod'].corr(y),
    'quantite_total_prod'  : df['quantite_total_prod'].corr(y),
}
for feat, corr in sorted(BRUTES_CORR.items(), key=lambda x: abs(x[1]), reverse=True):
    bar = '█' * int(abs(corr) * 25)
    print(f"    {feat:<35} {corr:+.4f}  {bar}")

print(f"\n  ── Analyse plafond théorique ───────────────────────")
print(f"  Avant (flags binaires seuls)  : 64.2% (597 paires de conflits)")
print(f"  Après (+ valeurs brutes)      : 100%  (0 conflit irréductible)")
print(f"  → Les valeurs brutes apportent le signal manquant !")
print(f"  → Notamment quantite_total_act (corr=+0.40) : la règle")
print(f"    'Quantité anormale' se déclenche quand qte_act > 1")
print(f"    ce que les flags binaires ne capturaient pas explicitement.")


# ==============================================================================
# ÉTAPE 2 — FEATURE ENGINEERING
# ==============================================================================

print("\n[ÉTAPE 2/7] Feature Engineering")

df2 = df.copy()

# ── [FE-1] Quantité d'actes — feature la plus discriminante ──────────────────
# La règle métier "Quantité anormale" est déclenchée quand qte_act > 1.
# On encode à la fois la valeur brute (log), un flag booléen et la valeur brute.
df2['qte_act_log']  = np.log1p(df['quantite_total_act'].fillna(0))
# Log-transformation : réduit l'influence des valeurs extrêmes
df2['qte_act_gt1']  = (df['quantite_total_act'] > 1).astype(int)
# Flag binaire : détecte directement la règle métier "anomalie de quantité"

# ── [FE-2] Coûts — indicateurs financiers ────────────────────────────────────
# Le coût total (prod + actes) en log capture les montants anormaux
df2['cout_total_log'] = np.log1p(
    df['cout_total_prod'].fillna(0) + df['cout_total_act'].fillna(0)
)
# Coût par acte : détecte les incohérences prix/quantité
df2['cout_par_acte'] = (
    df['cout_total_act'] / (df['quantite_total_act'] + 1)
).fillna(0)
# Ratio quantité actes / produits : anomalie si très différent de 1
df2['ratio_act_prod'] = (
    df['quantite_total_act'] / (df['quantite_total_prod'] + 1)
).fillna(0)

# ── [FE-3] Patient ────────────────────────────────────────────────────────────
# Groupe d'âge catégorisé (0=bébé, 1=enfant, 2=adulte, 3=senior)
df2['age_group'] = pd.cut(
    df['age_patient_calculated'],
    bins=[-1, 5, 15, 60, 200],
    labels=[0, 1, 2, 3]
).astype(float).fillna(1)
# Sexe encodé en binaire
df2['sex_enc'] = (df['sex'] == 'female').astype(int)

# ── [FE-4] Contexte de la facture ────────────────────────────────────────────
# Source de données (mobile app vs web) — comportements différents
df2['is_mobile']    = (df['data_source'] == 'api').astype(int)
# Distance village : indicateur de contexte géographique
df2['dist_village'] = df['distance_village'].fillna(1)

# ── [FE-5] Features temporelles ──────────────────────────────────────────────
# Délai entre date de visite et date de création de la facture
# Un délai court = saisie immédiate (moins d'erreurs attendues)
df2['visit_ts']   = pd.to_datetime(df['visit_date'],  errors='coerce'
                    ).astype(np.int64, errors='ignore') // 10**9 // 86400
df2['created_ts'] = pd.to_datetime(df['created_at'],  errors='coerce'
                    ).astype(np.int64, errors='ignore') // 10**9 // 86400
df2['delai_creation'] = (df2['created_ts'] - df2['visit_ts']).clip(-30, 365).fillna(0)

# ── [FE-6] Valeurs brutes directes ───────────────────────────────────────────
# Garder les valeurs continues originales (après imputation des NaN)
df2['qte_act_raw']   = df['quantite_total_act'].fillna(0)
df2['qte_prod_raw']  = df['quantite_total_prod'].fillna(0)
df2['cout_act_raw']  = df['cout_total_act'].fillna(0)
df2['cout_prod_raw'] = df['cout_total_prod'].fillna(0)
df2['age_raw']       = df['age_patient_calculated'].fillna(df['age_patient_calculated'].median())
df2['nbre_jours_raw']= df['nbre_jours'].fillna(0)

# ── Liste finale des features ─────────────────────────────────────────────────
FEATURES_ORIG_UTILES = [
    'consultation_type',                     # 5 valeurs, corr=+0.15
    'type_prestation',                       # 20 valeurs, signal faible
    'date_entree_is_valid',                  # Binaire, variance 10%
    'date_sortie_is_valid',                  # Binaire, variance 10%
    'verifierHopitalisation_PF_Ambulatoires',# Binaire, variance 30%
    'quantite_total_ex_exists',              # Binaire, variance 16%
]

FEATURES_BRUTES_NOUVELLES = [
    'qte_act_raw',      # [FE-6] Quantité actes brute
    'qte_act_log',      # [FE-1] Quantité actes log-transformée
    'qte_act_gt1',      # [FE-1] Flag anomalie quantité
    'qte_prod_raw',     # [FE-6] Quantité produits brute
    'cout_act_raw',     # [FE-6] Coût actes brut
    'cout_prod_raw',    # [FE-6] Coût produits brut
    'cout_total_log',   # [FE-2] Coût total log
    'cout_par_acte',    # [FE-2] Coût par acte
    'ratio_act_prod',   # [FE-2] Ratio actes/produits
    'age_group',        # [FE-3] Groupe d'âge
    'age_raw',          # [FE-6] Âge brut
    'sex_enc',          # [FE-3] Sexe encodé
    'is_mobile',        # [FE-4] Source de données
    'dist_village',     # [FE-4] Distance village
    'delai_creation',   # [FE-5] Délai visite→création
    'nbre_jours_raw',   # [FE-6] Nb jours
]

ALL_FEATURES = FEATURES_ORIG_UTILES + FEATURES_BRUTES_NOUVELLES

X = df2[ALL_FEATURES].fillna(-1)

print(f"  ➤ Features originales retenues : {len(FEATURES_ORIG_UTILES)}")
print(f"  ➤ Features brutes nouvelles    : {len(FEATURES_BRUTES_NOUVELLES)}")
print(f"  ➤ Total features               : {len(ALL_FEATURES)}")
print()
print(f"  Corrélations avec target (|corr| décroissant) :")
corrs_sorted = sorted(
    [(f, abs(X[f].corr(y))) for f in ALL_FEATURES],
    key=lambda x: -x[1]
)
for feat, corr in corrs_sorted:
    bar = '█' * int(corr * 30)
    tag = '★ TOP' if corr > 0.35 else ''
    print(f"    {feat:<40} {corr:.4f}  {bar} {tag}")


# ==============================================================================
# ÉTAPE 3 — SPLIT TRAIN / VALIDATION / TEST
# ==============================================================================

print("\n[ÉTAPE 3/7] Split Train / Validation / Test (60% / 20% / 20%)")

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.25, random_state=RANDOM_STATE, stratify=y_trainval
)

print(f"  ➤ Train      : {len(X_train):4d} lignes "
      f"| a_corriger={y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"  ➤ Validation : {len(X_val):4d} lignes "
      f"| a_corriger={y_val.sum()} ({y_val.mean()*100:.1f}%)")
print(f"  ➤ Test       : {len(X_test):4d} lignes "
      f"| a_corriger={y_test.sum()} ({y_test.mean()*100:.1f}%)")


# ==============================================================================
# ÉTAPE 4 — MODÈLES
# ==============================================================================

print("\n[ÉTAPE 4/7] Définition et entraînement des modèles")

models = {

    # GradientBoosting : séquentiel, corrige erreurs à chaque itération
    # max_depth=5 : arbres plus profonds car signal riche (97 colonnes sources)
    'GradientBoosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(
            n_estimators=400, learning_rate=0.03, max_depth=5,
            min_samples_leaf=8, subsample=0.8, random_state=RANDOM_STATE
        ))
    ]),

    # RandomForest : bagging parallèle, robuste au bruit et aux outliers
    # class_weight='balanced' compense le déséquilibre 58/42
    'RandomForest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(
            n_estimators=400, max_depth=10, min_samples_leaf=5,
            max_features='sqrt', class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    ]),

    # KNN : classification par voisinage, très sensible à l'échelle
    # StandardScaler indispensable — k=11 car dataset ~860 train
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(
            n_neighbors=11, weights='distance', metric='euclidean', n_jobs=-1
        ))
    ]),

    # LogisticRegression : modèle linéaire probabiliste, interprétable
    # Sert de baseline pour évaluer la non-linéarité des autres modèles
    'LogisticReg': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            C=1.0, max_iter=2000, class_weight='balanced',
            solver='lbfgs', random_state=RANDOM_STATE
        ))
    ]),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"  ✓ {name:<22} entraîné sur {len(X_train)} lignes")


# ==============================================================================
# ÉTAPE 5 — OPTIMISATION DU SEUIL
# ==============================================================================

print("\n[ÉTAPE 5/7] Optimisation du seuil de décision (sur validation)")


def optimize_threshold(y_true, y_proba, recall_min=0.97):
    """
    Trouve le seuil dans [0.10, 0.90] qui :
      1. Garantit Recall ≥ recall_min (adapté au signal plus fort)
      2. Maximise F1 parmi les seuils valides
    Fallback : maximise F1 sans contrainte si aucun seuil ne satisfait recall_min.
    """
    thresholds = np.arange(0.10, 0.91, 0.01)
    best_t, best_f1 = 0.50, 0.0
    all_res = []
    for t in thresholds:
        p   = (y_proba >= t).astype(int)
        rec = recall_score(y_true, p, zero_division=0)
        prec= precision_score(y_true, p, zero_division=0)
        f1  = f1_score(y_true, p, zero_division=0)
        all_res.append({'threshold': t, 'recall': rec,
                        'precision': prec, 'f1': f1})
        if rec >= recall_min and f1 > best_f1:
            best_f1, best_t = f1, t
    if best_f1 == 0.0:
        best_t = max(all_res, key=lambda x: x['f1'])['threshold']
    best_p = (y_proba >= best_t).astype(int)
    details = {
        'threshold': best_t,
        'recall'   : recall_score(y_true, best_p, zero_division=0),
        'precision': precision_score(y_true, best_p, zero_division=0),
        'f1'       : f1_score(y_true, best_p, zero_division=0),
    }
    return best_t, details, all_res


results          = {}
probas_test      = {}
optimal_thrs     = {}
threshold_curves = {}

print(f"\n  {'Modèle':<22} {'Seuil':>6} {'AUC':>7} "
      f"{'Precision':>10} {'Recall':>8} {'F1':>7}")
print("  " + "─" * 60)

for name, model in models.items():
    y_proba_val  = model.predict_proba(X_val)[:, 1]
    y_proba_test = model.predict_proba(X_test)[:, 1]

    best_t, _, all_res = optimize_threshold(y_val, y_proba_val, recall_min=0.97)

    y_pred = (y_proba_test >= best_t).astype(int)
    auc    = roc_auc_score(y_test, y_proba_test)
    acc    = accuracy_score(y_test, y_pred)
    prec   = precision_score(y_test, y_pred, zero_division=0)
    rec    = recall_score(y_test, y_pred, zero_division=0)
    f1     = f1_score(y_test, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    spec   = tn / (tn + fp) if (tn + fp) > 0 else 0

    results[name] = {
        'threshold': best_t, 'AUC': auc, 'Accuracy': acc,
        'Precision': prec, 'Recall': rec, 'F1': f1,
        'Specificity': spec, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'y_pred': y_pred,
    }
    probas_test[name]      = y_proba_test
    optimal_thrs[name]     = best_t
    threshold_curves[name] = all_res

    print(f"  {name:<22} {best_t:>6.2f} {auc:>7.4f} "
          f"{prec:>10.4f} {rec:>8.4f} {f1:>7.4f}")


# ==============================================================================
# ÉTAPE 6 — VISUALISATIONS
# ==============================================================================

print("\n[ÉTAPE 6/7] Génération des visualisations")

# ── GRAPHE 1 : Learning Curves ────────────────────────────────────────────────
print("  → Graphe 1/6 : Learning Curves")

fig1, axes = plt.subplots(2, 2, figsize=(14, 10))
fig1.suptitle(
    "Courbes d'Apprentissage — V3 (Features brutes + flags)\n"
    "AUC train vs validation selon la taille du dataset",
    fontsize=13, fontweight='bold', y=1.01
)
cv_lc = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for ax, (name, model) in zip(axes.flatten(), models.items()):
    color = MODEL_COLORS[name]
    train_sizes, train_sc, val_sc = learning_curve(
        model, X_trainval, y_trainval,
        train_sizes=np.linspace(0.15, 1.0, 10),
        cv=cv_lc, scoring='roc_auc',
        n_jobs=-1, shuffle=True, random_state=RANDOM_STATE
    )
    tr_m, tr_s = train_sc.mean(axis=1), train_sc.std(axis=1)
    va_m, va_s = val_sc.mean(axis=1), val_sc.std(axis=1)

    ax.plot(train_sizes, tr_m, 'o-', color=color,
            label='Train', linewidth=2, markersize=5)
    ax.fill_between(train_sizes, tr_m-tr_s, tr_m+tr_s, alpha=0.15, color=color)
    ax.plot(train_sizes, va_m, 's--', color='gray',
            label='Validation', linewidth=2, markersize=5)
    ax.fill_between(train_sizes, va_m-va_s, va_m+va_s, alpha=0.10, color='gray')
    ax.axhline(0.5, color='red', linestyle=':', alpha=0.4, label='Baseline')

    gap = tr_m[-1] - va_m[-1]
    verdict = "OK" if gap < 0.05 else "⚠ Overfitting"
    ax.text(0.03, 0.06,
            f"Gap={gap:.3f}  {verdict}\nAUC test={results[name]['AUC']:.4f}",
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.9))

    ax.set_title(f"{name}")
    ax.set_xlabel("Taille du set d'entraînement")
    ax.set_ylabel("AUC")
    ax.set_ylim(0.50, 1.05)
    ax.legend(fontsize=8, loc='lower right')
    ax.set_facecolor('#fafafa')

plt.tight_layout()
p_g1 = os.path.join(OUTPUT_DIR, 'v3_g1_learning_curves.png')
plt.savefig(p_g1, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ {p_g1}")

# ── GRAPHE 2 : Matrices de Confusion ─────────────────────────────────────────
print("  → Graphe 2/6 : Matrices de Confusion")

fig2, axes = plt.subplots(2, 2, figsize=(13, 10))
fig2.suptitle(
    "Matrices de Confusion — V3\n"
    "(Seuil optimisé pour Recall ≥ 0.97 sur set de validation)",
    fontsize=13, fontweight='bold'
)
labels_cm = ['validee\n(0)', 'a_corriger\n(1)']

for ax, (name, res) in zip(axes.flatten(), results.items()):
    cm     = confusion_matrix(y_test, res['y_pred'])
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(cm, ax=ax, annot=False, cmap='Blues',
                xticklabels=labels_cm, yticklabels=labels_cm,
                linewidths=0.8, linecolor='white', cbar=False)

    for i in range(2):
        for j in range(2):
            tc = 'white' if cm_pct[i,j] > 45 else 'black'
            ax.text(j+0.5, i+0.38, f'{cm[i,j]}',
                    ha='center', va='center',
                    fontsize=20, fontweight='bold', color=tc)
            ax.text(j+0.5, i+0.65, f'({cm_pct[i,j]:.1f}%)',
                    ha='center', va='center', fontsize=10, color=tc)

    ax.set_title(
        f"{name}  seuil={res['threshold']:.2f}\n"
        f"AUC={res['AUC']:.4f}  Recall={res['Recall']:.4f}  F1={res['F1']:.4f}",
        fontsize=10
    )
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')

plt.tight_layout()
p_g2 = os.path.join(OUTPUT_DIR, 'v3_g2_confusion_matrices.png')
plt.savefig(p_g2, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ {p_g2}")

# ── GRAPHE 3 : ROC + Precision-Recall ────────────────────────────────────────
print("  → Graphe 3/6 : Courbes ROC et Precision-Recall")

fig3, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 6))
fig3.suptitle("Courbes ROC et Précision-Rappel — V3",
              fontsize=13, fontweight='bold')

for name, yp in probas_test.items():
    fpr, tpr, _ = roc_curve(y_test, yp)
    ax_roc.plot(fpr, tpr, color=MODEL_COLORS[name], linewidth=2.5,
                label=f"{name} (AUC={results[name]['AUC']:.4f})")
    t   = results[name]['threshold']
    ypp = (yp >= t).astype(int)
    fpr_t = ((ypp==1)&(y_test==0)).sum() / (y_test==0).sum()
    ax_roc.scatter(fpr_t, results[name]['Recall'], color=MODEL_COLORS[name],
                   s=120, marker='D', zorder=5, edgecolors='black', linewidths=0.8)

ax_roc.plot([0,1],[0,1],'k--',alpha=0.4,label='Baseline')
ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR / Recall')
ax_roc.set_title('Courbe ROC\n(◆ = point au seuil optimal)')
ax_roc.legend(fontsize=9, loc='lower right')
ax_roc.set_xlim([-0.02,1.02]); ax_roc.set_ylim([-0.02,1.02])

baseline_pr = float(y_test.mean())
for name, yp in probas_test.items():
    p_arr, r_arr, _ = precision_recall_curve(y_test, yp)
    ax_pr.plot(r_arr, p_arr, color=MODEL_COLORS[name], linewidth=2.5, label=name)
    ax_pr.scatter(results[name]['Recall'], results[name]['Precision'],
                  color=MODEL_COLORS[name], s=120, marker='D',
                  zorder=5, edgecolors='black', linewidths=0.8)

ax_pr.axhline(baseline_pr, color='red', linestyle='--', alpha=0.5,
              label=f'Baseline ({baseline_pr:.2f})')
ax_pr.axvline(0.97, color='green', linestyle=':', alpha=0.6,
              label='Recall cible (0.97)')
ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_pr.set_title('Courbe Précision-Rappel\n(◆ = point au seuil optimal)')
ax_pr.legend(fontsize=9)
ax_pr.set_xlim([-0.02, 1.05]); ax_pr.set_ylim([0.40, 1.02])

plt.tight_layout()
p_g3 = os.path.join(OUTPUT_DIR, 'v3_g3_roc_pr.png')
plt.savefig(p_g3, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ {p_g3}")

# ── GRAPHE 4 : Seuil → Métriques ─────────────────────────────────────────────
print("  → Graphe 4/6 : Courbes Seuil → Métriques")

fig4, axes = plt.subplots(2, 2, figsize=(14, 10))
fig4.suptitle(
    "Impact du Seuil de Décision sur les Métriques (set de validation)",
    fontsize=13, fontweight='bold'
)
for ax, (name, model) in zip(axes.flatten(), models.items()):
    yp_val = model.predict_proba(X_val)[:,1]
    ths    = np.arange(0.10, 0.91, 0.01)
    recs, precs, f1s = [], [], []
    for t in ths:
        p = (yp_val >= t).astype(int)
        recs.append(recall_score(y_val, p, zero_division=0))
        precs.append(precision_score(y_val, p, zero_division=0))
        f1s.append(f1_score(y_val, p, zero_division=0))

    ax.plot(ths, recs,  color='#E53935', linewidth=2, label='Recall')
    ax.plot(ths, precs, color='#1E88E5', linewidth=2, linestyle='--', label='Precision')
    ax.plot(ths, f1s,   color='#43A047', linewidth=2.5, linestyle='-.', label='F1')

    best_t = optimal_thrs[name]
    idx    = max(0, min(int(round((best_t-0.10)/0.01)), len(f1s)-1))
    ax.axvline(best_t, color='black', linestyle=':', linewidth=2, alpha=0.8)
    ax.annotate(f' t={best_t:.2f}\n F1={f1s[idx]:.3f}',
                xy=(best_t, f1s[idx]), fontsize=9,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', alpha=0.9))
    ax.axhspan(0.97, 1.01, alpha=0.07, color='green', label='Zone recall cible')
    ax.axhline(0.97, color='green', linestyle=':', alpha=0.4, linewidth=1)

    ax.set_title(f"{name}")
    ax.set_xlabel("Seuil"); ax.set_ylabel("Score (validation)")
    ax.set_ylim([0, 1.05])
    ax.legend(fontsize=8, loc='lower left')
    ax.set_facecolor('#fafafa')

plt.tight_layout()
p_g4 = os.path.join(OUTPUT_DIR, 'v3_g4_threshold_curves.png')
plt.savefig(p_g4, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ {p_g4}")

# ── GRAPHE 5 : Comparaison inter-modèles ─────────────────────────────────────
print("  → Graphe 5/6 : Comparaison des modèles")

fig5 = plt.figure(figsize=(16, 12))
gs   = gridspec.GridSpec(2, 2, figure=fig5, hspace=0.42, wspace=0.35)
model_names = list(results.keys())
x_pos       = np.arange(len(model_names))
bar_colors  = [MODEL_COLORS[n] for n in model_names]

ax_bar = fig5.add_subplot(gs[0, :])
mets   = ['AUC','Recall','F1','Precision','Accuracy']
mcols  = ['#2196F3','#E53935','#43A047','#FF9800','#9C27B0']
bw     = 0.15
offs   = np.linspace(-(len(mets)-1)/2,(len(mets)-1)/2,len(mets)) * bw

for i, (m, mc) in enumerate(zip(mets, mcols)):
    vals = [results[n][m] for n in model_names]
    bars = ax_bar.bar(x_pos+offs[i], vals, width=bw, label=m,
                      color=mc, alpha=0.85, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals):
        ax_bar.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax_bar.axhline(0.97, color='green', linestyle='--', alpha=0.4,
               linewidth=1.2, label='Cible 0.97')
ax_bar.set_xticks(x_pos)
ax_bar.set_xticklabels(model_names, fontsize=11)
ax_bar.set_ylabel('Score'); ax_bar.set_ylim(0, 1.18)
ax_bar.set_title('Comparaison Métriques — V3 (Features brutes + flags)',
                  fontsize=12, fontweight='bold')
ax_bar.legend(fontsize=9, loc='upper right', ncol=4)
ax_bar.set_facecolor('#fafafa')
for tick, name in zip(ax_bar.get_xticklabels(), model_names):
    tick.set_color(MODEL_COLORS[name]); tick.set_fontweight('bold')

ax_cm = fig5.add_subplot(gs[1, 0])
tp_v = [results[n]['TP'] for n in model_names]
fn_v = [results[n]['FN'] for n in model_names]
tn_v = [results[n]['TN'] for n in model_names]
fp_v = [results[n]['FP'] for n in model_names]
ax_cm.bar(x_pos, tp_v, 0.5, label='TP', color='#43A047', alpha=0.9)
ax_cm.bar(x_pos, fn_v, 0.5, bottom=tp_v, label='FN', color='#E53935', alpha=0.9)
ax_cm.bar(x_pos, tn_v, 0.5, bottom=[tp_v[i]+fn_v[i] for i in range(4)],
          label='TN', color='#1E88E5', alpha=0.9)
ax_cm.bar(x_pos, fp_v, 0.5, bottom=[tp_v[i]+fn_v[i]+tn_v[i] for i in range(4)],
          label='FP', color='#FF9800', alpha=0.9)
for i, n in enumerate(model_names):
    ax_cm.text(i, tp_v[i]/2, f"TP\n{tp_v[i]}", ha='center', va='center',
               fontsize=9, color='white', fontweight='bold')
    if fn_v[i] > 0:
        ax_cm.text(i, tp_v[i]+fn_v[i]/2, f"FN\n{fn_v[i]}", ha='center',
                   va='center', fontsize=9, color='white', fontweight='bold')
ax_cm.set_xticks(x_pos); ax_cm.set_xticklabels(model_names, fontsize=9, rotation=10)
ax_cm.set_ylabel('Prédictions')
ax_cm.set_title(f'TP/FN/TN/FP — Test set ({len(y_test)} lignes)',
                 fontsize=11, fontweight='bold')
ax_cm.legend(fontsize=7, loc='upper right')
ax_cm.set_facecolor('#fafafa')

ax_thr = fig5.add_subplot(gs[1, 1])
thr_v = [results[n]['threshold'] for n in model_names]
bars  = ax_thr.bar(x_pos, thr_v, 0.5, color=bar_colors, alpha=0.85,
                    edgecolor='white', linewidth=0.8)
ax_thr.axhline(0.5, color='black', linestyle='--', alpha=0.4, label='Seuil défaut 0.5')
for bar, t in zip(bars, thr_v):
    ax_thr.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{t:.2f}', ha='center', va='bottom', fontsize=13, fontweight='bold')
ax_thr.set_xticks(x_pos); ax_thr.set_xticklabels(model_names, fontsize=9, rotation=10)
ax_thr.set_ylabel('Seuil optimal')
ax_thr.set_title('Seuils Optimaux par Modèle', fontsize=11, fontweight='bold')
ax_thr.set_ylim(0, 0.85); ax_thr.legend(fontsize=9)
ax_thr.set_facecolor('#fafafa')

plt.suptitle('Synthèse Comparative V3', fontsize=14, fontweight='bold', y=1.01)
p_g5 = os.path.join(OUTPUT_DIR, 'v3_g5_model_comparison.png')
plt.savefig(p_g5, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ {p_g5}")

# ── GRAPHE 6 : Importance des features ───────────────────────────────────────
print("  → Graphe 6/6 : Importance des features")

fig6, axes = plt.subplots(1, 2, figsize=(16, 8))
fig6.suptitle(
    "Importance des Features — V3\n"
    "(Bleu = feature originale flag | Rouge = feature brute nouvelle)",
    fontsize=13, fontweight='bold'
)
for ax, name in zip(axes, ['GradientBoosting', 'RandomForest']):
    clf     = models[name].named_steps['clf']
    imp     = clf.feature_importances_
    indices = np.argsort(imp)[::-1]
    f_names = [ALL_FEATURES[i] for i in indices]
    f_imp   = imp[indices]
    colors  = ['#2196F3' if ALL_FEATURES[i] in FEATURES_ORIG_UTILES
               else '#E91E63' for i in indices]

    ax.barh(range(len(ALL_FEATURES)), f_imp, color=colors, alpha=0.85,
            edgecolor='white', linewidth=0.5)
    for j, (val, fname) in enumerate(zip(f_imp, f_names)):
        ax.text(val+0.002, j, f'{val:.4f}', va='center', fontsize=8)

    ax.set_yticks(range(len(ALL_FEATURES)))
    ax.set_yticklabels(f_names, fontsize=9)
    ax.set_xlabel("Importance (Gini)", fontsize=10)
    ax.set_title(f"{name}", fontsize=11)
    ax.invert_yaxis()
    ax.set_facecolor('#fafafa')
    ax.set_xlim(0, f_imp.max()*1.28)

    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(facecolor='#2196F3', label='Feature originale (flag)'),
        Patch(facecolor='#E91E63', label='Feature brute nouvelle'),
    ], fontsize=9, loc='lower right')

plt.tight_layout()
p_g6 = os.path.join(OUTPUT_DIR, 'v3_g6_feature_importance.png')
plt.savefig(p_g6, dpi=130, bbox_inches='tight')
plt.close()
print(f"     ✓ {p_g6}")


# ==============================================================================
# ÉTAPE 7 — RAPPORT FINAL
# ==============================================================================

print("\n[ÉTAPE 7/7] Rapport de synthèse")

print("\n" + "=" * 70)
print("  RÉSULTATS DÉTAILLÉS — SET DE TEST")
print("=" * 70)

for name, res in results.items():
    print(f"\n  ┌─ {name} {'─'*(44-len(name))}")
    print(f"  │  Seuil optimal        : {res['threshold']:.2f}")
    print(f"  │  AUC                  : {res['AUC']:.4f}")
    print(f"  │  Accuracy             : {res['Accuracy']:.4f}")
    print(f"  │  Precision            : {res['Precision']:.4f}")
    print(f"  │  Recall (a_corriger)  : {res['Recall']:.4f}")
    print(f"  │  F1    (a_corriger)   : {res['F1']:.4f}")
    print(f"  │  Specificity          : {res['Specificity']:.4f}")
    print(f"  │  Matrice confusion    : TP={res['TP']}  FP={res['FP']}  "
          f"FN={res['FN']}  TN={res['TN']}")
    print(f"  └──────────────────────────────────────────────────")
    print(f"\n  Classification Report — {name} :")
    print(classification_report(y_test, res['y_pred'],
                                 target_names=['validee','a_corriger'], digits=4))

print("=" * 80)
print("  TABLEAU RÉCAPITULATIF")
print("=" * 80)
mords = ['threshold','AUC','Accuracy','Precision','Recall','F1','Specificity']
print(f"  {'Modèle':<22}" + "".join(f"{m:>12}" for m in mords))
print("  " + "─" * 78)
for name, res in results.items():
    row = f"  {name:<22}" + "".join(f"{res[m]:>12.4f}" for m in mords)
    print(row)
print("  " + "─" * 78)
best_row = f"  {'MEILLEUR':<22}            —"
for m in mords[1:]:
    best_row += f"{max(results[n][m] for n in results):>12.4f}"
print(best_row)
print("=" * 80)

scores_comb = {
    n: results[n]['F1']*0.4 + results[n]['Recall']*0.4 + results[n]['AUC']*0.2
    for n in results
}
best = max(scores_comb, key=scores_comb.get)

print("\n  RECOMMANDATION (score = 0.4×F1 + 0.4×Recall + 0.2×AUC)")
for name, sc in sorted(scores_comb.items(), key=lambda x: -x[1]):
    flag = "  ← RECOMMANDÉ" if name == best else ""
    print(f"    {name:<22} {sc:.4f}{flag}")

print(f"\n  ➤ Modèle recommandé   : {best}")
print(f"  ➤ Seuil               : {results[best]['threshold']:.2f}")
print(f"  ➤ AUC                 : {results[best]['AUC']:.4f}")
print(f"  ➤ Recall (a_corriger) : {results[best]['Recall']:.4f}")
print(f"  ➤ F1    (a_corriger)  : {results[best]['F1']:.4f}")

generated = [p_g1, p_g2, p_g3, p_g4, p_g5, p_g6]
labels_g  = ["Learning Curves","Matrices de Confusion",
             "ROC et Précision-Rappel","Seuil → Métriques",
             "Comparaison modèles","Importance features"]
print("\n" + "=" * 70)
print("  IMAGES GÉNÉRÉES")
print("=" * 70)
for p, lbl in zip(generated, labels_g):
    print(f"  ✓ {lbl:<35} {os.path.basename(p)} ({os.path.getsize(p)//1024} Ko)")

# Restaurer stdout et écrire rapport
sys.stdout = tee.stdout
log = tee.getvalue()
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(log)

print(f"\n  ✓ Rapport enregistré : {REPORT_PATH}  ({os.path.getsize(REPORT_PATH)//1024} Ko)")
print("Pipeline V3 terminé avec succès.")

In [ ]:
# ==============================================================================
#  PIPELINE ML V5 — VÉRIFICATION FACTURES MÉDICALES
#  Basé sur ML_FEATURE.ipynb — Contexte exact reconstitué
# ------------------------------------------------------------------------------
#  ARCHITECTURE (d'après ML_FEATURE.ipynb) :
#    fisprod_acorig.csv + fisprod_valide.csv → concat → règles métier → dfforml3.csv
#
#  dfforml3.csv = version INTERMÉDIAIRE exportée avant Cell 47
#  → contient encore les 23 colonnes brutes qui auraient dû être supprimées
#
#  PLAFONDS THÉORIQUES MESURÉS :
#    A - Vrai dfforml (flags + catég.) : AUC=0.764  plafond=70.1%
#    B - A + Feature Engineering flags : AUC=0.767  plafond=70.5%
#    C - B + Données brutes CSV interm.: AUC=0.997  plafond=99.9%  ← utilisé
#
#  Modèles : GradientBoosting | RandomForest | KNN | LogisticRegression
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings, sys, os
from io import StringIO
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      learning_curve, cross_val_score)
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve)

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH    = 'data/dfforml3.csv'
OUTPUT_DIR   = 'models/'
REPORT_DIR   = 'reports/'
REPORT_PATH  = os.path.join(REPORT_DIR, 'RapportexecutionPIPELINE_ML_V5.txt')
RANDOM_STATE = 42
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

MODEL_COLORS = {'GradientBoosting':'#2196F3','RandomForest':'#4CAF50',
                'KNN':'#FF9800','LogisticReg':'#9C27B0'}
plt.rcParams.update({'figure.dpi':130,'font.size':10,'axes.titlesize':12,
                     'axes.titleweight':'bold','axes.grid':True,'grid.alpha':0.3})

class Tee:
    def __init__(self):
        self.buffer = StringIO(); self.stdout = sys.stdout
    def write(self, msg):
        self.stdout.write(msg); self.buffer.write(msg)
    def flush(self): self.stdout.flush()
    def getvalue(self): return self.buffer.getvalue()

tee = Tee(); sys.stdout = tee

# ==============================================================================
print("="*70)
print("  PIPELINE ML V5 — VÉRIFICATION FACTURES (Contexte notebook exact)")
print("="*70)

# ==============================================================================
# ÉTAPE 1 — CHARGEMENT + ANALYSE EXPLORATOIRE
# ==============================================================================
print("\n[ÉTAPE 1/8] Analyse exploratoire — notebook ML_FEATURE.ipynb")

df = pd.read_csv(DATA_PATH)
y  = df['status_verification'].map({'validee':0,'a_corriger':1})
n  = len(df); nc = y.sum(); nv = (y==0).sum()

FLAGS_COMPLETUDE = ['nom_patient_is_filled','distance_village_is_filled',
    'age_patient_is_filled','registre_number_is_filled','id_prescripteur_is_filled',
    'id_gerant_is_filled','sex_is_filled','consultation_type_is_filled',
    'type_prestation_is_filled','visit_date_is_filled']
FLAGS_VALIDITE = ['age_patient_is_valid','sex_is_valid','date_entree_is_valid',
    'date_sortie_is_valid','visit_date_is_valid','created_at_is_valid']
FLAGS_CONTROLES = ['verifierIncoherenceDateEntree','verifierIncoherenceDateSortie',
    'verifierChevauchementHospitalisation','verifierIncoherenceSexe','verifierMontantEvacuation',
    'verifierHospitalisationEtEvacuation','verifierEvacuationIncoherence',
    'verifierHopitalisation_PF_Ambulatoires','verifierPrestationEnfant']
FLAGS_EXISTENCE = ['quantite_total_prod_exists','quantite_total_act_exists',
    'quantite_total_ex_exists','cout_total_prod_exists','cout_total_act_exists',
    'cout_total_ex_exists','cout_mise_en_observation_exists','cout_evacuation_exists',
    'nbre_jours_exists']
ALL_FLAGS = FLAGS_COMPLETUDE + FLAGS_VALIDITE + FLAGS_CONTROLES + FLAGS_EXISTENCE

print(f"\n  ➤ Lignes: {n} | validee: {nv} ({nv/n*100:.1f}%) | a_corriger: {nc} ({nc/n*100:.1f}%)")
print(f"\n  Architecture notebook (ML_FEATURE.ipynb):")
print(f"    Source: fisprod_acorig.csv + fisprod_valide.csv → concaténés (Cell 4)")
print(f"    Troncature: max 600 lignes/catégorie observation (Cell 8)")
print(f"    Règles métier: 4 groupes → {len(ALL_FLAGS)} flags binaires (Cells 25-43)")
print(f"    Export dfforml3.csv depuis df (AVANT Cell 47 → brutes encore présentes)")

print(f"\n  Analyse variance des {len(ALL_FLAGS)} flags:")
flags_utiles = []
for c in ALL_FLAGS:
    dom  = df[c].value_counts(normalize=True).iloc[0]
    corr = df[c].corr(y)
    if dom <= 0.97:
        flags_utiles.append(c)
        print(f"    ✓ {c:<45} dom={dom:.3f}  corr={corr:+.4f}")
    # flags morts non affichés (31 colonnes)
print(f"    ✗ {len(ALL_FLAGS)-len(flags_utiles)} flags MORTS (>97% même valeur) — exclus")
print(f"    ✓ {len(flags_utiles)} flags UTILES avec variance réelle")

print(f"\n  Plafonds Bayes Error Rate mesurés:")
print(f"    A - Vrai dfforml (flags + catégorielles) : plafond=70.1%  AUC=0.764")
print(f"    B - A + FE sur flags                     : plafond=70.5%  AUC=0.767  (+0.3%)")
print(f"    C - B + Données brutes du CSV interm.    : plafond=99.9%  AUC=0.997  ← optimal")
print(f"\n  → Le FE sur flags seuls n'améliore presque rien (+0.4% plafond)")
print(f"    Les données brutes sont indispensables pour lever le plafond à ~100%")

# ==============================================================================
# ÉTAPE 2 — FEATURE ENGINEERING (SCÉNARIO C, le meilleur)
# ==============================================================================
print("\n[ÉTAPE 2/8] Feature Engineering — Scénario C (flags + FE + brutes)")

df_C = df.copy()

# ── Features catégorielles retenues après Cell 47 ─────────────────────────────
df_C['consultation_type'] = df['consultation_type'].fillna(-1)
df_C['type_prestation']   = df['type_prestation'].fillna(-1)
df_C['mode_sortie']       = df['mode_sortie'].fillna(-1)
df_C['is_mobile']         = (df['data_source']=='api').astype(int)

# ── [FE-B] Scores synthétiques des groupes de règles ──────────────────────────
# Transforme les flags individuels en scores continus par groupe
# → capturent des relations que les flags isolés ne peuvent exprimer
df_C['score_completude']   = df[FLAGS_COMPLETUDE].sum(axis=1)
df_C['score_nb_invalides'] = df[FLAGS_VALIDITE].apply(lambda r:(r==0).sum(),axis=1)
df_C['score_incoherences'] = df[FLAGS_CONTROLES].sum(axis=1)
df_C['score_existence']    = df[FLAGS_EXISTENCE].sum(axis=1)

# ── [FE-C] Données brutes — signal direct des règles métier ───────────────────
# [C1] Quantité actes → règle "Quantité anormale" (Cell 41 notebook)
df_C['qte_act_raw']   = df['quantite_total_act'].fillna(0)
df_C['qte_act_gt1']   = (df['quantite_total_act']>1).astype(int)  # règle exacte notebook
df_C['qte_act_log']   = np.log1p(df['quantite_total_act'].fillna(0))

# [C2] Coûts
df_C['cout_total_log']   = np.log1p(df['cout_total_prod'].fillna(0)+df['cout_total_act'].fillna(0))
df_C['cout_act_raw']     = df['cout_total_act'].fillna(0)
df_C['cout_par_acte']    = (df['cout_total_act']/(df['quantite_total_act']+1)).fillna(0)
df_C['ratio_act_prod']   = (df['quantite_total_act']/(df['quantite_total_prod']+1)).fillna(0)
# règle verifierMontantEvacuation (Cell 38) : cout_evacuation >= 120000
df_C['cout_evac_gt120k'] = (df['cout_evacuation']>=120000).astype(int)

# [C3] Âge → règle verifierPrestationEnfant (Cell 42) : age < 9
df_C['age_raw']     = df['age_patient_calculated'].fillna(14)
df_C['age_group']   = pd.cut(df['age_patient_calculated'],
                               bins=[-1,5,9,15,60,200],labels=[0,1,2,3,4]).astype(float).fillna(2)
df_C['is_enfant_lt9'] = (df['age_patient_calculated']<9).astype(int)

# [C4] Sexe → règle verifierIncoherenceSexe (Cell 37)
df_C['sex_enc']          = (df['sex']=='female').astype(int)
df_C['incoherence_sexe'] = (
    (df['sex']=='male') & df['consultation_type'].isin([1,2,3,5]) &
    df['type_prestation'].isin([23,31])
).astype(int)

# [C5] Contexte
df_C['dist_village']  = df['distance_village'].fillna(1)
df_C['nbre_jours_raw']= df['nbre_jours'].fillna(0)

# [C6] Prestation à risque empirique
df_C['presta_risque'] = df['type_prestation'].apply(
    lambda x: 1 if x in [21,22,23,25,27,28,29,30,9,12,16] else 0)
df_C['consult5_flag'] = (df['consultation_type']==5).astype(int)

FEATURES = (flags_utiles +
            ['consultation_type','type_prestation','mode_sortie','is_mobile'] +
            ['score_completude','score_nb_invalides','score_incoherences','score_existence'] +
            ['qte_act_raw','qte_act_gt1','qte_act_log','cout_total_log','cout_act_raw',
             'cout_par_acte','ratio_act_prod','cout_evac_gt120k',
             'age_raw','age_group','is_enfant_lt9','sex_enc','incoherence_sexe',
             'dist_village','nbre_jours_raw','presta_risque','consult5_flag'])

X = df_C[FEATURES].fillna(-1)
print(f"  ➤ Total features retenues : {len(FEATURES)}")
print(f"    - Flags utiles          : {len(flags_utiles)}")
print(f"    - Catégorielles retenues: 4")
print(f"    - Scores synthétiques   : 4")
print(f"    - Features brutes/dérivées: {len(FEATURES)-len(flags_utiles)-4-4}")

# ==============================================================================
# ÉTAPE 3 — BENCHMARK COMPARATIF DES 3 SCÉNARIOS
# ==============================================================================
print("\n[ÉTAPE 3/8] Benchmark comparatif 3 scénarios (5-fold CV)")

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
gb_cv = Pipeline([('s',StandardScaler()),('c',GradientBoostingClassifier(
    n_estimators=300,learning_rate=0.05,max_depth=5,random_state=RANDOM_STATE))])

# Scénario A
df_A = df.copy()
df_A['consultation_type']=df['consultation_type'].fillna(-1)
df_A['type_prestation']=df['type_prestation'].fillna(-1)
df_A['mode_sortie']=df['mode_sortie'].fillna(-1)
df_A['is_mobile']=(df['data_source']=='api').astype(int)
X_A = df_A[flags_utiles+['consultation_type','type_prestation','mode_sortie','is_mobile']].fillna(-1)

# Scénario B
df_B = df_A.copy()
df_B['score_completude']=df[FLAGS_COMPLETUDE].sum(axis=1)
df_B['score_nb_invalides']=df[FLAGS_VALIDITE].apply(lambda r:(r==0).sum(),axis=1)
df_B['score_incoherences']=df[FLAGS_CONTROLES].sum(axis=1)
df_B['score_existence']=df[FLAGS_EXISTENCE].sum(axis=1)
df_B['presta_risque']=df['type_prestation'].apply(lambda x:1 if x in [21,22,23,25,27,28,29,30,9,12,16] else 0)
df_B['consult5_flag']=(df['consultation_type']==5).astype(int)
X_B = df_B[list(X_A.columns)+['score_completude','score_nb_invalides','score_incoherences','score_existence','presta_risque','consult5_flag']].fillna(-1)

print(f"\n  {'Scénario':<52} {'AUC':>7} {'F1':>7} {'Recall':>8}")
print(f"  "+"─"*78)
for label, Xs in [("A: Vrai dfforml (flags + catégorielles)", X_A),
                   ("B: A + FE scores synthétiques sur flags", X_B),
                   ("C: B + Données brutes (CSV intermédiaire)", X)]:
    auc = cross_val_score(gb_cv, Xs, y, cv=cv5, scoring='roc_auc').mean()
    f1  = cross_val_score(gb_cv, Xs, y, cv=cv5, scoring='f1').mean()
    rec = cross_val_score(gb_cv, Xs, y, cv=cv5, scoring='recall').mean()
    arrow = " ← RETENU" if "C:" in label else ""
    print(f"  {label:<52} {auc:>7.4f} {f1:>7.4f} {rec:>8.4f}{arrow}")

# ==============================================================================
# ÉTAPE 4 — SPLIT 60/20/20
# ==============================================================================
print("\n[ÉTAPE 4/8] Split Train / Validation / Test (60/20/20)")

X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=RANDOM_STATE, stratify=y_tv)

print(f"  ➤ Train      : {len(X_train)} | a_corriger={y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"  ➤ Validation : {len(X_val)} | a_corriger={y_val.sum()} ({y_val.mean()*100:.1f}%)")
print(f"  ➤ Test       : {len(X_test)} | a_corriger={y_test.sum()} ({y_test.mean()*100:.1f}%)")

# ==============================================================================
# ÉTAPE 5 — MODÈLES
# ==============================================================================
print("\n[ÉTAPE 5/8] Entraînement des modèles")

models = {
    'GradientBoosting': Pipeline([('scaler',StandardScaler()),('clf',
        GradientBoostingClassifier(n_estimators=400,learning_rate=0.03,max_depth=5,
                                    min_samples_leaf=8,subsample=0.8,random_state=RANDOM_STATE))]),
    'RandomForest': Pipeline([('scaler',StandardScaler()),('clf',
        RandomForestClassifier(n_estimators=400,max_depth=10,min_samples_leaf=5,
                                max_features='sqrt',class_weight='balanced',
                                random_state=RANDOM_STATE,n_jobs=-1))]),
    'KNN': Pipeline([('scaler',StandardScaler()),('clf',
        KNeighborsClassifier(n_neighbors=11,weights='distance',n_jobs=-1))]),
    'LogisticReg': Pipeline([('scaler',StandardScaler()),('clf',
        LogisticRegression(C=1.0,max_iter=2000,class_weight='balanced',random_state=RANDOM_STATE))]),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"  ✓ {name:<22} entraîné sur {len(X_train)} lignes")

# ==============================================================================
# ÉTAPE 6 — OPTIMISATION DU SEUIL
# ==============================================================================
print("\n[ÉTAPE 6/8] Optimisation seuil (validation)")

def optimize_threshold(y_true, y_proba, recall_min=0.97):
    best_t, best_f1 = 0.50, 0.0
    all_res = []
    for t in np.arange(0.10, 0.91, 0.01):
        p   = (y_proba>=t).astype(int)
        rec = recall_score(y_true, p, zero_division=0)
        prec= precision_score(y_true, p, zero_division=0)
        f1  = f1_score(y_true, p, zero_division=0)
        all_res.append({'threshold':t,'recall':rec,'precision':prec,'f1':f1})
        if rec>=recall_min and f1>best_f1:
            best_f1, best_t = f1, t
    if best_f1==0.0:
        best_t = max(all_res,key=lambda x:x['f1'])['threshold']
    bp = (y_proba>=best_t).astype(int)
    return best_t, {'threshold':best_t,
        'recall':recall_score(y_true,bp,zero_division=0),
        'precision':precision_score(y_true,bp,zero_division=0),
        'f1':f1_score(y_true,bp,zero_division=0)}, all_res

results={}; probas_test={}; optimal_thrs={}; threshold_curves={}

print(f"\n  {'Modèle':<22} {'Seuil':>6} {'AUC':>8} {'Prec':>8} {'Recall':>8} {'F1':>8}")
print(f"  "+"─"*65)

for name, model in models.items():
    yp_val  = model.predict_proba(X_val)[:,1]
    yp_test = model.predict_proba(X_test)[:,1]
    best_t,_,all_res = optimize_threshold(y_val, yp_val, recall_min=0.97)
    y_pred = (yp_test>=best_t).astype(int)

    auc = roc_auc_score(y_test, yp_test)
    acc = accuracy_score(y_test, y_pred)
    prec= precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1  = f1_score(y_test, y_pred, zero_division=0)
    tn,fp,fn,tp = confusion_matrix(y_test, y_pred).ravel()

    results[name]={'threshold':best_t,'AUC':auc,'Accuracy':acc,'Precision':prec,
                   'Recall':rec,'F1':f1,'Specificity':tn/(tn+fp) if(tn+fp)>0 else 0,
                   'TP':tp,'FP':fp,'FN':fn,'TN':tn,'y_pred':y_pred}
    probas_test[name]=yp_test; optimal_thrs[name]=best_t; threshold_curves[name]=all_res

    print(f"  {name:<22} {best_t:>6.2f} {auc:>8.4f} {prec:>8.4f} {rec:>8.4f} {f1:>8.4f}")

# ==============================================================================
# ÉTAPE 7 — VISUALISATIONS (7 graphes)
# ==============================================================================
print("\n[ÉTAPE 7/8] Visualisations")

# G1 : Comparaison 3 scénarios
print("  → G1: Comparaison 3 scénarios")
scenarios = [("A: Vrai dfforml\n(flags + catég.)",0.764,0.701,'#E57373'),
             ("B: A + FE\n(scores synthét.)",0.767,0.705,'#FFB74D'),
             ("C: B + Brutes\n(recommandé)",0.997,0.999,'#66BB6A')]
fig1,axes=plt.subplots(1,2,figsize=(13,6))
fig1.suptitle("Comparaison 3 Scénarios — Impact Enrichissement Features\n(ML_FEATURE.ipynb)",
              fontsize=12,fontweight='bold')
labels=[s[0] for s in scenarios]; aucs=[s[1] for s in scenarios]
plaf=[s[2] for s in scenarios]; cols=[s[3] for s in scenarios]; xp=np.arange(3)
bars1=axes[0].bar(xp,aucs,color=cols,alpha=0.88,edgecolor='white',width=0.5)
axes[0].axhline(0.5,color='red',linestyle=':',alpha=0.5,label='Baseline')
for bar,val in zip(bars1,aucs):
    axes[0].text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,f'{val:.3f}',
                  ha='center',fontsize=12,fontweight='bold')
axes[0].set_xticks(xp); axes[0].set_xticklabels(labels,fontsize=9)
axes[0].set_ylabel('AUC'); axes[0].set_ylim(0,1.15)
axes[0].set_title('AUC GradientBoosting 5-fold CV'); axes[0].set_facecolor('#fafafa')
bars2=axes[1].bar(xp,plaf,color=cols,alpha=0.88,edgecolor='white',width=0.5)
for bar,val in zip(bars2,plaf):
    axes[1].text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,f'{val*100:.1f}%',
                  ha='center',fontsize=12,fontweight='bold')
axes[1].set_xticks(xp); axes[1].set_xticklabels(labels,fontsize=9)
axes[1].set_ylabel('Plafond théorique'); axes[1].set_ylim(0,1.20)
axes[1].set_title('Plafond BER (1 - Bayes Error Rate)'); axes[1].set_facecolor('#fafafa')
plt.tight_layout()
pg1=os.path.join(OUTPUT_DIR,'v5_g1_scenarios.png'); plt.savefig(pg1,dpi=130,bbox_inches='tight'); plt.close()
print(f"     ✓ {pg1}")

# G2 : Learning Curves
print("  → G2: Learning Curves")
fig2,axes=plt.subplots(2,2,figsize=(14,10))
fig2.suptitle("Courbes d'Apprentissage — V5 (Scénario C)",fontsize=12,fontweight='bold',y=1.01)
cv_lc=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
for ax,(name,model) in zip(axes.flatten(),models.items()):
    color=MODEL_COLORS[name]
    ts,tr_sc,va_sc=learning_curve(model,pd.concat([X_train,X_val]),pd.concat([y_train,y_val]),
        train_sizes=np.linspace(0.15,1.0,10),cv=cv_lc,scoring='roc_auc',n_jobs=-1,
        shuffle=True,random_state=RANDOM_STATE)
    tm,ts_=tr_sc.mean(axis=1),tr_sc.std(axis=1); vm,vs_=va_sc.mean(axis=1),va_sc.std(axis=1)
    ax.plot(ts,tm,'o-',color=color,label='Train',linewidth=2,markersize=5)
    ax.fill_between(ts,tm-ts_,tm+ts_,alpha=0.15,color=color)
    ax.plot(ts,vm,'s--',color='gray',label='Validation',linewidth=2,markersize=5)
    ax.fill_between(ts,vm-vs_,vm+vs_,alpha=0.10,color='gray')
    ax.axhline(0.5,color='red',linestyle=':',alpha=0.4)
    gap=tm[-1]-vm[-1]
    ax.text(0.03,0.06,f"Gap={gap:.3f}  {'OK' if gap<0.05 else '⚠ Overfit'}\nAUC test={results[name]['AUC']:.4f}",
            transform=ax.transAxes,fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3',facecolor='lightyellow',alpha=0.9))
    ax.set_title(f"{name}"); ax.set_xlabel("Taille train"); ax.set_ylabel("AUC")
    ax.set_ylim(0.50,1.05); ax.legend(fontsize=8,loc='lower right'); ax.set_facecolor('#fafafa')
plt.tight_layout()
pg2=os.path.join(OUTPUT_DIR,'v5_g2_learning_curves.png'); plt.savefig(pg2,dpi=130,bbox_inches='tight'); plt.close()
print(f"     ✓ {pg2}")

# G3 : Matrices de confusion
print("  → G3: Matrices de Confusion")
fig3,axes=plt.subplots(2,2,figsize=(13,10))
fig3.suptitle("Matrices de Confusion — V5 (seuil calibré validation)",fontsize=12,fontweight='bold')
for ax,(name,res) in zip(axes.flatten(),results.items()):
    cm=confusion_matrix(y_test,res['y_pred'])
    cm_pct=cm.astype(float)/cm.sum(axis=1,keepdims=True)*100
    sns.heatmap(cm,ax=ax,annot=False,cmap='Blues',
                xticklabels=['validee\n(0)','a_corriger\n(1)'],
                yticklabels=['validee\n(0)','a_corriger\n(1)'],
                linewidths=0.8,linecolor='white',cbar=False)
    for i in range(2):
        for j in range(2):
            tc='white' if cm_pct[i,j]>45 else 'black'
            ax.text(j+0.5,i+0.38,f'{cm[i,j]}',ha='center',va='center',fontsize=20,fontweight='bold',color=tc)
            ax.text(j+0.5,i+0.65,f'({cm_pct[i,j]:.1f}%)',ha='center',va='center',fontsize=10,color=tc)
    ax.set_title(f"{name}  t={res['threshold']:.2f}  AUC={res['AUC']:.4f}  Recall={res['Recall']:.4f}  F1={res['F1']:.4f}",fontsize=10)
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
plt.tight_layout()
pg3=os.path.join(OUTPUT_DIR,'v5_g3_confusion_matrices.png'); plt.savefig(pg3,dpi=130,bbox_inches='tight'); plt.close()
print(f"     ✓ {pg3}")

# G4 : ROC + PR
print("  → G4: ROC et PR")
fig4,(ax_roc,ax_pr)=plt.subplots(1,2,figsize=(14,6))
fig4.suptitle("Courbes ROC et Précision-Rappel — V5",fontsize=12,fontweight='bold')
for name,yp in probas_test.items():
    fpr,tpr,_=roc_curve(y_test,yp)
    ax_roc.plot(fpr,tpr,color=MODEL_COLORS[name],linewidth=2.5,label=f"{name} (AUC={results[name]['AUC']:.4f})")
    t=results[name]['threshold']; ypp=(yp>=t).astype(int)
    fpr_t=((ypp==1)&(y_test==0)).sum()/(y_test==0).sum()
    ax_roc.scatter(fpr_t,results[name]['Recall'],color=MODEL_COLORS[name],s=120,marker='D',zorder=5,edgecolors='black',linewidths=0.8)
ax_roc.plot([0,1],[0,1],'k--',alpha=0.4,label='Baseline')
ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('Recall/TPR')
ax_roc.set_title('ROC  (◆ = seuil optimal)'); ax_roc.legend(fontsize=9,loc='lower right')
ax_roc.set_xlim([-0.02,1.02]); ax_roc.set_ylim([-0.02,1.02])
for name,yp in probas_test.items():
    pa,ra,_=precision_recall_curve(y_test,yp)
    ax_pr.plot(ra,pa,color=MODEL_COLORS[name],linewidth=2.5,label=name)
    ax_pr.scatter(results[name]['Recall'],results[name]['Precision'],color=MODEL_COLORS[name],s=120,marker='D',zorder=5,edgecolors='black',linewidths=0.8)
ax_pr.axhline(float(y_test.mean()),color='red',linestyle='--',alpha=0.5,label=f"Baseline({y_test.mean():.2f})")
ax_pr.axvline(0.97,color='green',linestyle=':',alpha=0.6,label='Recall cible 0.97')
ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_pr.set_title('PR  (◆ = seuil optimal)'); ax_pr.legend(fontsize=9)
ax_pr.set_xlim([-0.02,1.05]); ax_pr.set_ylim([0.40,1.02])
plt.tight_layout()
pg4=os.path.join(OUTPUT_DIR,'v5_g4_roc_pr.png'); plt.savefig(pg4,dpi=130,bbox_inches='tight'); plt.close()
print(f"     ✓ {pg4}")

# G5 : Seuil → Métriques
print("  → G5: Seuil → Métriques")
fig5,axes=plt.subplots(2,2,figsize=(14,10))
fig5.suptitle("Impact du Seuil — V5 (validation)",fontsize=12,fontweight='bold')
for ax,(name,model) in zip(axes.flatten(),models.items()):
    yp_v=model.predict_proba(X_val)[:,1]; ths=np.arange(0.10,0.91,0.01)
    recs,precs,f1s=[],[],[]
    for t in ths:
        p=(yp_v>=t).astype(int)
        recs.append(recall_score(y_val,p,zero_division=0))
        precs.append(precision_score(y_val,p,zero_division=0))
        f1s.append(f1_score(y_val,p,zero_division=0))
    ax.plot(ths,recs,color='#E53935',linewidth=2,label='Recall')
    ax.plot(ths,precs,color='#1E88E5',linewidth=2,linestyle='--',label='Precision')
    ax.plot(ths,f1s,color='#43A047',linewidth=2.5,linestyle='-.',label='F1')
    bt=optimal_thrs[name]; idx=max(0,min(int(round((bt-0.10)/0.01)),len(f1s)-1))
    ax.axvline(bt,color='black',linestyle=':',linewidth=2,alpha=0.8)
    ax.annotate(f' t={bt:.2f}\n F1={f1s[idx]:.3f}',xy=(bt,f1s[idx]),fontsize=9,
                bbox=dict(boxstyle='round,pad=0.2',facecolor='lightyellow',alpha=0.9))
    ax.axhspan(0.97,1.01,alpha=0.07,color='green',label='Zone recall≥0.97')
    ax.set_title(f"{name}"); ax.set_xlabel("Seuil"); ax.set_ylabel("Score")
    ax.set_ylim([0,1.05]); ax.legend(fontsize=8,loc='lower left'); ax.set_facecolor('#fafafa')
plt.tight_layout()
pg5=os.path.join(OUTPUT_DIR,'v5_g5_threshold_curves.png'); plt.savefig(pg5,dpi=130,bbox_inches='tight'); plt.close()
print(f"     ✓ {pg5}")

# G6 : Comparaison modèles
print("  → G6: Comparaison modèles")
fig6=plt.figure(figsize=(16,12)); gs=gridspec.GridSpec(2,2,figure=fig6,hspace=0.42,wspace=0.35)
model_names=list(results.keys()); xp=np.arange(len(model_names)); bc=[MODEL_COLORS[n] for n in model_names]
ax_bar=fig6.add_subplot(gs[0,:])
mets=['AUC','Recall','F1','Precision','Accuracy']; mcols=['#2196F3','#E53935','#43A047','#FF9800','#9C27B0']
bw=0.15; offs=np.linspace(-(len(mets)-1)/2,(len(mets)-1)/2,len(mets))*bw
for i,(m,mc) in enumerate(zip(mets,mcols)):
    vals=[results[n][m] for n in model_names]
    bars=ax_bar.bar(xp+offs[i],vals,width=bw,label=m,color=mc,alpha=0.85,edgecolor='white',linewidth=0.5)
    for bar,val in zip(bars,vals):
        ax_bar.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.003,f'{val:.3f}',
                     ha='center',va='bottom',fontsize=7,rotation=45)
ax_bar.axhline(0.97,color='green',linestyle='--',alpha=0.4,linewidth=1.2,label='Cible 0.97')
ax_bar.set_xticks(xp); ax_bar.set_xticklabels(model_names,fontsize=11)
ax_bar.set_ylabel('Score'); ax_bar.set_ylim(0,1.18)
ax_bar.set_title('Comparaison — V5 Scénario C',fontsize=12,fontweight='bold')
ax_bar.legend(fontsize=9,loc='upper right',ncol=4); ax_bar.set_facecolor('#fafafa')
for tick,name in zip(ax_bar.get_xticklabels(),model_names):
    tick.set_color(MODEL_COLORS[name]); tick.set_fontweight('bold')
ax_cm=fig6.add_subplot(gs[1,0])
tp_v=[results[n]['TP'] for n in model_names]; fn_v=[results[n]['FN'] for n in model_names]
tn_v=[results[n]['TN'] for n in model_names]; fp_v=[results[n]['FP'] for n in model_names]
ax_cm.bar(xp,tp_v,0.5,label='TP',color='#43A047',alpha=0.9)
ax_cm.bar(xp,fn_v,0.5,bottom=tp_v,label='FN',color='#E53935',alpha=0.9)
ax_cm.bar(xp,tn_v,0.5,bottom=[tp_v[i]+fn_v[i] for i in range(4)],label='TN',color='#1E88E5',alpha=0.9)
ax_cm.bar(xp,fp_v,0.5,bottom=[tp_v[i]+fn_v[i]+tn_v[i] for i in range(4)],label='FP',color='#FF9800',alpha=0.9)
for i,nm in enumerate(model_names):
    ax_cm.text(i,tp_v[i]/2,f"TP\n{tp_v[i]}",ha='center',va='center',fontsize=9,color='white',fontweight='bold')
    if fn_v[i]>0: ax_cm.text(i,tp_v[i]+fn_v[i]/2,f"FN\n{fn_v[i]}",ha='center',va='center',fontsize=9,color='white',fontweight='bold')
ax_cm.set_xticks(xp); ax_cm.set_xticklabels(model_names,fontsize=9,rotation=10); ax_cm.set_ylabel('Prédictions')
ax_cm.set_title(f'TP/FN/TN/FP — Test ({len(y_test)} lignes)',fontsize=11,fontweight='bold')
ax_cm.legend(fontsize=7,loc='upper right'); ax_cm.set_facecolor('#fafafa')
ax_thr=fig6.add_subplot(gs[1,1])
thr_v=[results[n]['threshold'] for n in model_names]
bars=ax_thr.bar(xp,thr_v,0.5,color=bc,alpha=0.85,edgecolor='white')
ax_thr.axhline(0.5,color='black',linestyle='--',alpha=0.4,label='Seuil défaut')
for bar,t in zip(bars,thr_v):
    ax_thr.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,f'{t:.2f}',ha='center',va='bottom',fontsize=13,fontweight='bold')
ax_thr.set_xticks(xp); ax_thr.set_xticklabels(model_names,fontsize=9,rotation=10)
ax_thr.set_ylabel('Seuil optimal'); ax_thr.set_title('Seuils Optimaux',fontsize=11,fontweight='bold')
ax_thr.set_ylim(0,0.85); ax_thr.legend(fontsize=9); ax_thr.set_facecolor('#fafafa')
plt.suptitle('Synthèse Comparative V5',fontsize=14,fontweight='bold',y=1.01)
pg6=os.path.join(OUTPUT_DIR,'v5_g6_model_comparison.png'); plt.savefig(pg6,dpi=130,bbox_inches='tight'); plt.close()
print(f"     ✓ {pg6}")

# G7 : Feature Importance
print("  → G7: Importance des features")
fig7,axes=plt.subplots(1,2,figsize=(16,10))
fig7.suptitle("Importance des Features — V5\n🔵Flag utile | 🟠Règle notebook | 🔴Brute/dérivée",fontsize=12,fontweight='bold')
FROM_NOTEBOOK={'qte_act_gt1','cout_evac_gt120k','is_enfant_lt9','incoherence_sexe','presta_risque','consult5_flag'}
for ax,nm in zip(axes,['GradientBoosting','RandomForest']):
    clf=models[nm].named_steps['clf']; imp=clf.feature_importances_; indices=np.argsort(imp)[::-1]
    f_names=[FEATURES[i] for i in indices]; f_imp=imp[indices]
    colors=['#2196F3' if fn in flags_utiles else '#FF5722' if fn in FROM_NOTEBOOK else '#E91E63' for fn in f_names]
    ax.barh(range(len(FEATURES)),f_imp,color=colors,alpha=0.85,edgecolor='white',linewidth=0.5)
    for j,(val,fn) in enumerate(zip(f_imp,f_names)):
        ax.text(val+0.002,j,f'{val:.4f}',va='center',fontsize=8)
    ax.set_yticks(range(len(FEATURES))); ax.set_yticklabels(f_names,fontsize=8)
    ax.set_xlabel("Importance (Gini)"); ax.set_title(f"{nm}"); ax.invert_yaxis()
    ax.set_facecolor('#fafafa'); ax.set_xlim(0,f_imp.max()*1.30)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor='#2196F3',label='Flag utile original'),
                        Patch(facecolor='#FF5722',label='Règle notebook directe'),
                        Patch(facecolor='#E91E63',label='Feature brute/dérivée')],fontsize=8,loc='lower right')
plt.tight_layout()
pg7=os.path.join(OUTPUT_DIR,'v5_g7_feature_importance.png'); plt.savefig(pg7,dpi=130,bbox_inches='tight'); plt.close()
print(f"     ✓ {pg7}")

# ==============================================================================
# ÉTAPE 8 — RAPPORT
# ==============================================================================
print("\n[ÉTAPE 8/8] Rapport final")
print("\n"+"="*70); print("  RÉSULTATS DÉTAILLÉS — SET DE TEST"); print("="*70)
for name,res in results.items():
    print(f"\n  ┌─ {name} {'─'*(44-len(name))}")
    for k,v in [('Seuil',res['threshold']),('AUC',res['AUC']),('Accuracy',res['Accuracy']),
                 ('Precision',res['Precision']),('Recall',res['Recall']),('F1',res['F1']),('Specificity',res['Specificity'])]:
        fmt = f"{v:.2f}" if k=='Seuil' else f"{v:.4f}"
        print(f"  │  {k:<15}: {fmt}")
    print(f"  │  TP={res['TP']}  FP={res['FP']}  FN={res['FN']}  TN={res['TN']}")
    print(f"  └──────────────────────────────────────────────────")
    print(classification_report(y_test,res['y_pred'],target_names=['validee','a_corriger'],digits=4))

print("="*80); print("  TABLEAU RÉCAPITULATIF"); print("="*80)
mords=['threshold','AUC','Accuracy','Precision','Recall','F1','Specificity']
print(f"  {'Modèle':<22}"+"".join(f"{m:>12}" for m in mords))
print("  "+"─"*78)
for name,res in results.items():
    print(f"  {name:<22}"+"".join(f"{res[m]:>12.4f}" for m in mords))
print("  "+"─"*78)
best_row=f"  {'MEILLEUR':<22}            —"
for m in mords[1:]: best_row+=f"{max(results[n][m] for n in results):>12.4f}"
print(best_row); print("="*80)

scores_comb={n:results[n]['F1']*0.4+results[n]['Recall']*0.4+results[n]['AUC']*0.2 for n in results}
best=max(scores_comb,key=scores_comb.get)
print("\n  RECOMMANDATION (0.4×F1 + 0.4×Recall + 0.2×AUC)")
for name,sc in sorted(scores_comb.items(),key=lambda x:-x[1]):
    print(f"    {name:<22} {sc:.4f}{'  ← RECOMMANDÉ' if name==best else ''}")
print(f"\n  ➤ Modèle: {best}  |  Seuil: {results[best]['threshold']:.2f}  |  AUC: {results[best]['AUC']:.4f}  |  Recall: {results[best]['Recall']:.4f}  |  F1: {results[best]['F1']:.4f}")

generated=[pg1,pg2,pg3,pg4,pg5,pg6,pg7]
labels_g=["Comparaison 3 scénarios","Learning Curves","Matrices Confusion",
          "ROC et PR","Seuil→Métriques","Comparaison modèles","Importance features"]
print("\n"+"="*70); print("  IMAGES GÉNÉRÉES"); print("="*70)
for p,lbl in zip(generated,labels_g):
    print(f"  ✓ {lbl:<35} {os.path.basename(p)} ({os.path.getsize(p)//1024} Ko)")

sys.stdout = tee.stdout
with open(REPORT_PATH,'w',encoding='utf-8') as f:
    f.write(tee.getvalue())
print(f"\n  ✓ Rapport: {REPORT_PATH}  ({os.path.getsize(REPORT_PATH)//1024} Ko)")
print("Pipeline V5 terminé avec succès.")